# Nanophos: aggregate DVP phosphoproteomics of HEK293 cells

HEK293 cells expressing A53T α-synuclein, seeded with α-synuclein preformed fibrils (PFF) or treated with PBS, without or with bafilomycin A1 (BafA1). Fractions of the cells were cut by laser microdissection; each lysate was split 1:9 into a proteome and a phosphoproteome sample (nanoPhos phosphopeptide enrichment), and both were searched with Spectronaut. Each phosphosite is normalised to its parent protein measured in the same lysate. This notebook makes the panels of Figure 6 and Figures S9 and S10.

| Short | Meaning |
|---|---|
| CWO | cells without aggregates |
| CW | cells with aggregates |
| EA | excised aggregates (obs value "Pure Aggregates") |
| RC | rest cells: the cells with aggregates after the aggregate was excised |
| PFF, PBS | seeded with preformed fibrils; PBS control |
| −BafA1, +BafA1 | without, with bafilomycin A1, an inhibitor of the lysosomal V-ATPase |
| Class I | phosphosite localised with probability ≥ 0.75 |
| kNN | k-nearest-neighbour imputation |
| eBayes | moderated t-test with empirical Bayes variance moderation (limma) |
| log2FC | log2 fold change |
| FDR | false discovery rate, Benjamini-Hochberg within each contrast |
| CV | coefficient of variation |
| PCA | principal component analysis |

**Inputs** (paths in the first cell of section 2): the Spectronaut phosphopeptide, proteome and fragment reports (`input_phospho`, `input_prot`, `input_frag`; PRIDE), the search FASTA (`fasta_path`; entry names and gene symbols) and the sample annotations (`obs_metadata_path`).

**Outputs:** result tables in `output_dir` (one empirical Bayes table per mode and contrast, the detection table of section 10); figure panels in the folders of the first cell of section 2 (`figures/`), written by `save_panel` and assembled in section 14.

# 1. Imports

In [ ]:
# --- standard library ---
import logging
import os
import re
import sys
import warnings
from datetime import date

# --- scientific stack ---
import anndata as ad
import numpy as np
import pandas as pd
from scipy import optimize as _sopt, stats as _sps

# --- plotting ---
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, NullLocator
from matplotlib.transforms import blended_transform_factory
from adjustText import adjust_text

# --- alphapepttools ---
import alphapepttools as at
from alphapepttools.pl.figure import create_figure, label_axes
from alphapepttools.pl.plots import label_plot

# --- project modules in src/ ---
# The path is relative to this notebook's folder, which VS Code uses as the
# working directory of a notebook. Importing figure_style sets the matplotlib and
# alphapepttools style of every panel. The preprocessing of section 2 is the
# collapse_precursors (PeptideCollapse_v4.py) and nanophos_preprocessing.py.
sys.path.insert(0, "src")
from figure_style import (ANNOT_FS, AXES_LW, BG_COLOR, DOWN_COLOR, DRAW_SCALE, GAP_MM,
                          GRID_COLS, GRID_ROWS, HIGHLIGHT_COLOR, LABEL_FS, LEGEND_FS,
                          LINE_LW, MARKER_LW, MM_PER_IN, REF_COLOR, SCALE, THIN_LW,
                          UNIT_H_MM, UP_COLOR, compose_figure, figsize, native_pt,
                          save_panel, shade, style_summary)
from PeptideCollapse_v4 import collapse_precursors
from nanophos_preprocessing import (
    META_COLS, apply_condition_aware_classI_mask, contaminant_groups, filter_per_condition,
    filter_proteins, hybrid_impute, normalize_phospho_by_condition_mean,
    group_gene, peptide_var, phospho_adata, pivot_wide, proteome_adata,
    read_fasta_entries, site_var,
)

logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
today = date.today().strftime("%Y%m%d")      # date stamp of the output file names


# 2. Preprocessing

From the Spectronaut reports to the AnnData objects of the analysis: contaminant protein groups removed, precursors collapsed to phosphosites and to modified peptides, blank and outlier wells dropped, Class I localisation decided per condition, completeness filter, imputation, and normalisation of every phospho feature to its parent protein.

## 2.1 Reports

In [ ]:
# Every path of the notebook, relative to the repository root.
input_phospho = 'data/Nanophos/20260716_160621_MaOe_nanoPhos_aggregates_new_fasta_Report_Denys_ms2_level.tsv'   # Spectronaut phosphopeptide report, precursor level
input_prot    = 'data/Nanophos/20260905_154107_MaOe_proteome_aggregates_new_fasta_Report.tsv'      # Spectronaut proteome report
input_frag    = 'data/Nanophos/20260923_204246_MaOe_nanoPhos_aggregates_new_fasta_fragments_Report.tsv'    # Spectronaut fragment report (section 12)
fasta_path    = 'data/Nanophos/260522_Human_CAN_20416entries_ASYN_A53T.fasta'   # search FASTA: entry names and gene symbols
obs_metadata_path = 'data/Nanophos/metadata.csv'                                 # well -> condition and sample annotations
output_dir    = 'results/Nanophos/'                         # result tables
# Panel folders: save_panel writes into them, section 14 assembles them.
FIG6_DIR = "figures/Figure6_Nanophos"
SI9_DIR  = "figures/SIs/SI9_Nanophos"         # Figure S9: QC, volcano +BafA1, detection heatmaps
SI10_DIR = "figures/SIs/SI10_Nanophos_SNCA"   # Figure S10: C-terminal SNCA +BafA1, y ions, enzymes
os.makedirs(output_dir, exist_ok=True)


In [ ]:
df_phospho  = pd.read_csv(input_phospho, sep='\t')
df_proteome = pd.read_csv(input_prot,       sep='\t')
print(f"phospho:  {df_phospho.shape}")
print(f"proteome: {df_proteome.shape}")

In [ ]:
# Contaminants, by the rule of the other notebooks (flag_contaminants in
# src/preprocessing.py): protein groups whose accessions all come from the
# contaminant FASTA ('Cont_'), keratins, immunoglobulins, and the skin, serum and
# laboratory proteins of its list. Entry names and gene symbols come from the
# search FASTA; the proteome report has no gene column. The flagged groups leave
# both reports before any other step.
fasta_entries = read_fasta_entries(fasta_path)
cont_flags = contaminant_groups(
    pd.concat([df_phospho["PG.ProteinGroups"], df_proteome["PG.ProteinGroups"]]), fasta_entries)
contaminants = set(cont_flags.index[cont_flags["is_contaminant"]])

_n_rows, _n_groups = len(df_phospho), len(df_proteome)
df_phospho  = df_phospho[~df_phospho["PG.ProteinGroups"].astype(str).isin(contaminants)].copy()
df_proteome = df_proteome[~df_proteome["PG.ProteinGroups"].astype(str).isin(contaminants)].copy()
_flagged = cont_flags.loc[list(contaminants)]
print(f"contaminant protein groups: {len(contaminants)} (flags overlap: "
      + ", ".join(f"{k} {int(_flagged[k].sum())}" for k in
                  ("is_crap", "is_contaminant_keratin", "is_immunoglobulin", "is_extra_contaminant"))
      + ")")
print(f"phospho:  {_n_rows - len(df_phospho):,} of {_n_rows:,} precursor rows removed")
print(f"proteome: {_n_groups - len(df_proteome):,} of {_n_groups:,} protein groups removed")


## 2.2 Phosphosites and modified peptides; blank and outlier wells

In [ ]:
# One pass over the precursor report gives both matrices, log2, with the log2
# values 0 and 1 (quantities 1 and 2) set to missing:
#   modified peptides  precursors with the same sequence and the same set of
#                      modifications (charge states; phosphates on different
#                      residues of the sequence) summed per run
#   phosphosites       per site and multiplicity, the median over the
#                      precursors that carry the site
# No localisation filter here: the Class I decision is made per condition in
# 2.3, from the localisation probability of every site in every run
# (loc_per_run).
df_peptides_raw, df_sites_raw, loc_per_run = collapse_precursors(df_phospho)
print(f"unmasked sites:    {len(df_sites_raw):,}")
print(f"unmasked peptides: {len(df_peptides_raw):,}")
print(f"loc matrix:        {loc_per_run.shape}")

In [ ]:
# Sample filter, decided on the site matrix and applied to both matrices:
#   (a) blank wells, and conditions with fewer than 3 wells;
#   (b) outlier wells: fewer Class I sites (loc >= 0.75 in that run) than 0.60 x
#       the median of their condition.
META = set(META_COLS)
PEPTIDE_META = {                      # metadata columns of the peptide matrix
    'peptide_collapse_key', 'PG.Genes', 'PG.ProteinGroups',
    'clean_sequence', 'all_modifications', 'phospho_count',
    'EG.PTMAssayProbability',
}

_cond_pat = re.compile(r'Phospho_[A-P]\d{2}_(.+?)(?:_\d+shapes)?$')
_well_pat = re.compile(r'Phospho_([A-P]\d{2})_')

def parse_condition(fname):
    m = _cond_pat.search(str(fname))
    return m.group(1) if m else None

# (a) blanks and small conditions
sample_cols_raw = [c for c in df_sites_raw.columns if c not in META]
cond_per_sample = pd.Series({c: parse_condition(c) for c in sample_cols_raw})
blank_cols = cond_per_sample[
    cond_per_sample.isna() | cond_per_sample.str.contains('Blank', case=False, na=False)
].index.tolist()
df_sites_raw_noblank = df_sites_raw.drop(columns=blank_cols)
s2c = cond_per_sample.drop(blank_cols)
cond_n = s2c.value_counts()
small = cond_n[cond_n < 3].index.tolist()
if small:
    drop_small = s2c[s2c.isin(small)].index.tolist()
    df_sites_raw_noblank = df_sites_raw_noblank.drop(columns=drop_small)
    s2c = s2c.drop(drop_small)
else:
    drop_small = []

# (b) Class I sites per well -> outlier wells
sample_cols_nb = [c for c in df_sites_raw_noblank.columns if c not in META]
loc_noblank    = loc_per_run[sample_cols_nb]
classI_per_sample = (loc_noblank >= 0.75).sum(axis=0)

samples_df = pd.DataFrame({
    'sample':    classI_per_sample.index,
    'ids':       classI_per_sample.values,
    'condition': classI_per_sample.index.map(s2c),
}).dropna(subset=['condition'])

hard_outliers = []
for cond, grp in samples_df.groupby('condition'):
    med = grp['ids'].median()
    hard_outliers.extend(grp[grp['ids'] < 0.60 * med]['sample'].tolist())

df_sites_raw_noblank = df_sites_raw_noblank.drop(columns=hard_outliers)
s2c = s2c.drop(hard_outliers)
outlier_wells = [_well_pat.search(s).group(1) for s in hard_outliers]

# (c) the same wells out of the peptide matrix
peptide_drops = blank_cols + drop_small + hard_outliers
df_peptides_raw_noblank = df_peptides_raw.drop(
    columns=[c for c in peptide_drops if c in df_peptides_raw.columns],
)

print(f"sites    noblank: {df_sites_raw_noblank.shape}")
print(f"peptides noblank: {df_peptides_raw_noblank.shape}")
print(f"dropped outlier wells: {outlier_wells}")
print(f"\nsurviving conditions:\n{s2c.value_counts().sort_index()}")

## 2.3 Class I sites and completeness filter

In [ ]:
# Class I, decided per (site, condition): if >= 50 % of the replicates localise
# the site with probability >= 0.75, every value of the site in that condition is
# kept, including replicates below the cutoff; otherwise only the localised
# values. Then sites measured in >= 70 % of the replicates of at least one condition.
df_sites_CI    = apply_condition_aware_classI_mask(
    df_sites_raw_noblank, loc_per_run, s2c,
    classI_cutoff=0.75, condition_threshold=0.50,
)
df_sites_CI_70 = filter_per_condition(df_sites_CI, s2c, threshold=0.70, mode='any')
print(f"Class I sites, 70 % filter: {len(df_sites_CI_70):,} sites")

In [ ]:
# Modified-peptide matrix, same completeness filter. It is used with its
# observed values only: the SNCA C-terminal phosphopeptide (Figure 6d, S10a) and
# the enzyme panels (Figure S10 e, f).
df_peptides_70 = filter_per_condition(
    df_peptides_raw_noblank, s2c,
    threshold=0.70, mode='any',
    meta_cols=PEPTIDE_META,
)
print(f"Modified peptides, 70 % filter: {len(df_peptides_70):,} peptides")

## 2.4 Imputation

In [ ]:
# Samples as rows. Imputation per (site, condition): measured in >= 50 % of the
# replicates -> the missing ones by kNN; measured in <= 1 replicate -> drawn from
# a Gaussian down-shifted by 1.8 SD, width 0.3 SD, of that well's values (seed
# 42); in between they stay missing. The peptide matrix is not imputed.
phos_site_wide_obs = pivot_wide(df_sites_CI_70, 'PTM_Collapse_key', META, _well_pat)
phos_site_wide_imp, _ = hybrid_impute(
    phos_site_wide_obs, s2c,
    sample_id_col='sample_id', quant_substring='~',
    knn_min_frac=0.5, mec_max_count=1, verbose=False,
)

phos_peptide_wide_obs = pivot_wide(df_peptides_70, 'peptide_collapse_key', PEPTIDE_META, _well_pat)

print("Pivot + impute complete:")
print(f"  phos_site:     obs {phos_site_wide_obs.shape}     imp {phos_site_wide_imp.shape}")
print(f"  phos_peptide:  obs {phos_peptide_wide_obs.shape}")

In [ ]:
# Proteome: log2 PG.Quantity per well, the wells of the phospho matrix (blanks,
# conditions with fewer than 3 wells and the phospho outlier wells dropped),
# protein groups quantified in >= 70 % of the replicates of at least one
# condition, imputed as the phosphosites.
def _clean_col(c):
    m = re.match(r'^\[\d+\]\s+(.+)\.raw\.PG\.Quantity$', str(c))
    return m.group(1) if m else c

prot = df_proteome.copy()
prot.columns = [_clean_col(c) for c in prot.columns]

prot_meta = ['PG.ProteinGroups', 'PG.Genes']
prot_sample_cols = [c for c in prot.columns if c not in prot_meta]

prot_long = prot.set_index('PG.ProteinGroups')[prot_sample_cols].T
prot_long = prot_long.loc[:, prot_long.notna().any(axis=0)]
prot_long = np.log2(prot_long.replace(0, np.nan))

_well_pat_prot = re.compile(r'Proteomes_([A-P]\d{2})_')
_cond_pat_prot = re.compile(r'Proteomes_[A-P]\d{2}_(.+?)(?:_\d+shapes)?$')
prot_long['well']      = [_well_pat_prot.search(s).group(1) for s in prot_long.index]
prot_long['condition'] = [_cond_pat_prot.search(s).group(1) for s in prot_long.index]

blank_mask = prot_long['condition'].str.contains('Blank', case=False, na=False)
prot_long = prot_long.loc[~blank_mask & ~prot_long['well'].isin(outlier_wells)].copy()
cn = prot_long['condition'].value_counts()
small = cn[cn < 3].index.tolist()
if small:
    prot_long = prot_long.loc[~prot_long['condition'].isin(small)].copy()

prot_meta_cols = ['well', 'condition']
prot_filt = filter_proteins(prot_long, meta_cols=prot_meta_cols,
                            how='condition', cutoff=0.70, condition_col='condition')

# sample -> condition for the proteome (its file names differ from the phospho ones)
prot_s2c = prot_filt['condition']

prot_imp, _ = hybrid_impute(
    prot_filt,
    prot_s2c,
    meta_cols=prot_meta_cols,
    sample_id_col=None,        # sample IDs are in prot_filt's index
    knn_min_frac=0.5,
    mec_max_count=1,
)
print(f"prot_imp shape: {prot_imp.shape}")

## 2.5 Normalisation to the parent protein

In [ ]:
# Every phospho feature minus the mean log2 intensity of its parent protein over
# the wells of its condition, so the value is the phospho intensity per unit of
# parent protein. The condition mean and not the well's own protein value:
# the proteome is a separate injection, so pairing by well does not cancel the
# loading of the phospho run and adds the noise of the proteome run. Features
# without a quantified parent protein are dropped.
phos_site_wide_obs_norm,    map_site_obs    = normalize_phospho_by_condition_mean(phos_site_wide_obs,    prot_filt)
phos_site_wide_imp_norm,    map_site_imp    = normalize_phospho_by_condition_mean(phos_site_wide_imp,    prot_imp)
phos_peptide_wide_obs_norm, map_peptide_obs = normalize_phospho_by_condition_mean(phos_peptide_wide_obs, prot_filt)

print("Normalization complete:")
print(f"  site:     {phos_site_wide_imp_norm.shape}     (mapped {len(map_site_imp):,})")
print(f"  peptide:  {phos_peptide_wide_obs_norm.shape}    (mapped {len(map_peptide_obs):,})")

## 2.6 AnnData objects

In [ ]:
# The AnnData objects of the analysis cells, one row per well, obs from
# metadata.csv:
#   adata_phos_site_obs_raw / _obs_norm   Class I sites, observed (detection, eBayes non_imputed)
#   adata_phos_site_imp_raw / _imp_norm   Class I sites, imputed (rank plot, PCA / eBayes)
#   adata_phos_peptide_obs_norm           modified peptides, observed, normalised
#   adata_prot_imp                        proteome, imputed
# _raw = not normalised to the protein. adata_phos_raw and adata_prot are the
# names the figure cells use.

# --- obs_metadata, indexed by well ---
obs_metadata_raw = pd.read_csv(obs_metadata_path, sep=',', index_col=0)
obs_metadata = (obs_metadata_raw
    .rename_axis('sample_id_proteome').reset_index().set_index('evotip_position'))
obs_metadata.index.name = None
obs_metadata['evotip_position'] = obs_metadata.index.astype(str)
for col in ('class', 'class_clean', 'class_renamed', 'class_shapes',
            'Sample Type', 'Aggregate', 'Inhibitor', 'PFF', 'Treatment',
            'CellType_Aggregate', 'Disease_CellType_Aggregate'):
    if col in obs_metadata.columns:
        obs_metadata[col] = obs_metadata[col].astype('category')

# --- the phospho and proteome matrices hold the same wells, all in the metadata ---
phos_wells = set(phos_site_wide_imp['well'])
prot_wells = set(prot_imp['well'])
assert phos_wells == prot_wells, "well mismatch"
all_wells = sorted(phos_wells)
missing = set(all_wells) - set(obs_metadata.index)
assert not missing, f"metadata missing for: {missing}"

# --- var tables ---
var_site_all = site_var(
    sorted(c for c in phos_site_wide_obs.columns if '~' in str(c)),
    set(map_site_imp) | set(map_site_obs),
)
var_peptide_all = peptide_var(
    sorted(c for c in phos_peptide_wide_obs.columns if '~' in str(c)),
    df_peptides_raw_noblank,
    set(map_peptide_obs),
)

# --- AnnData objects ---
adata_phos_site_obs_raw  = phospho_adata(phos_site_wide_obs,      var_site_all, all_wells, obs_metadata,
    'site-level (Class I), pre-imputation, NOT proteome-normalized')
adata_phos_site_obs_norm = phospho_adata(phos_site_wide_obs_norm, var_site_all, all_wells, obs_metadata,
    'site-level (Class I), pre-imputation, proteome-normalized')
adata_phos_site_imp_raw  = phospho_adata(phos_site_wide_imp,      var_site_all, all_wells, obs_metadata,
    'site-level (Class I), hybrid-imputed, NOT proteome-normalized')
adata_phos_site_imp_norm = phospho_adata(phos_site_wide_imp_norm, var_site_all, all_wells, obs_metadata,
    'site-level (Class I), hybrid-imputed, proteome-normalized')
adata_phos_peptide_obs_norm = phospho_adata(phos_peptide_wide_obs_norm, var_peptide_all, all_wells, obs_metadata,
    'peptide-level, pre-imputation, proteome-normalized')
adata_prot_imp = proteome_adata(prot_imp, all_wells, obs_metadata, 'proteome, hybrid-imputed')

adata_phos_raw = adata_phos_site_imp_raw
adata_prot     = adata_prot_imp

all_adatas = {
    'adata_phos_site_obs_raw': adata_phos_site_obs_raw,
    'adata_phos_site_obs_norm': adata_phos_site_obs_norm,
    'adata_phos_site_imp_raw': adata_phos_site_imp_raw,
    'adata_phos_site_imp_norm': adata_phos_site_imp_norm,
    'adata_phos_peptide_obs_norm': adata_phos_peptide_obs_norm,
    'adata_prot_imp': adata_prot_imp,
}
print(f"Built {len(all_adatas)} adatas:")
for name, a in all_adatas.items():
    n_nan = int(np.isnan(a.X).sum())
    pct = 100 * n_nan / max(1, a.X.size)
    print(f"  {name:<35}  {a.shape}  NaN={n_nan:>7,} ({pct:>4.1f}%)")

## 2.7 Completeness rule for the contrasts

In [ ]:
# A site enters a contrast of section 8 if it has measured (not imputed) values
# in >= 60 % and in >= 2 of the replicates of both groups (AND filter, the
# reported results) or of at least one group (OR filter).
MIN_FRAC_REAL  = 0.60
MIN_COUNT_REAL = 2

def _min_real(n_samples):
    return max(MIN_COUNT_REAL, int(np.ceil(MIN_FRAC_REAL * n_samples)))

def sites_with_enough_values(group1, group2, pre_wide, adata_for_obs, require,
                             biological_col='class_renamed'):
    """Features with >= _min_real(n) measured values in both groups
    (require="both") or in at least one of them (require="either")."""
    if require not in ("both", "either"):
        raise ValueError(f"require must be 'both' or 'either', got {require!r}")
    feature_cols = [c for c in pre_wide.columns if '~' in str(c)]
    pre_by_well = pre_wide.set_index('well')[feature_cols]
    g1_wells = adata_for_obs.obs.index[adata_for_obs.obs[biological_col] == group1].tolist()
    g2_wells = adata_for_obs.obs.index[adata_for_obs.obs[biological_col] == group2].tolist()
    if not g1_wells or not g2_wells:
        return []
    ok1 = pre_by_well.loc[g1_wells].notna().sum(axis=0) >= _min_real(len(g1_wells))
    ok2 = pre_by_well.loc[g2_wells].notna().sum(axis=0) >= _min_real(len(g2_wells))
    keep = (ok1 & ok2) if require == "both" else (ok1 | ok2)
    return pre_by_well.columns[keep].tolist()

def _sanitize(s):
    """Condition label -> a name patsy accepts ('+' -> 'Plus', '-' -> 'Minus', ' ' -> '_')."""
    return str(s).replace("+", "Plus").replace("-", "Minus").replace(" ", "_")

# 3. Style, colours and condition order

In [ ]:
# Page grid, fonts, line widths, colour roles (UP_COLOR, DOWN_COLOR, BG_COLOR,
# HIGHLIGHT_COLOR, REF_COLOR), shade, save_panel and compose_figure come from
# src/figure_style.py (imported in section 1), shared by all notebooks of the
# project. This cell holds what belongs to the nanophos data: condition order,
# colours and short labels.
print(style_summary())

# ---------------------------------------------------------------- CONDITIONS
biological = "class_renamed"

# Conditions, colours and short labels as in the 1_HEK notebook: the same
# HEK293 system and the same ten conditions, so a condition has one colour in
# every figure of the paper.

# Plotting order: PBS -> cells without -> cells with -> excised aggregates ->
# rest cells, -BafA1 before +BafA1 within each.
biological_order = [
    "-BafA1 PBS Cells Without Aggregates",
    "+BafA1 PBS Cells Without Aggregates",
    "-BafA1 PFF Cells Without Aggregates",
    "+BafA1 PFF Cells Without Aggregates",
    "-BafA1 PFF Cells With Aggregates",
    "+BafA1 PFF Cells With Aggregates",
    "-BafA1 PFF Pure Aggregates",
    "+BafA1 PFF Pure Aggregates",
    "-BafA1 PFF Rest Cells",
    "+BafA1 PFF Rest Cells",
]

# Fraction colour. Keys are the obs values: the data say "Pure Aggregates",
# displayed as EA. The first three colours are Paul Tol's colour-blind-safe
# "light" scheme; EA is darker because violet moves toward blue under
# deuteranopia, and lightness keeps it apart from CWO.
FRACTION_COLORS = {
    "Cells Without Aggregates": "#7fb3d8",   # blue
    "Cells With Aggregates":    "#e8a07a",   # orange
    "Rest Cells":               "#8cc5b2",   # teal
    "Pure Aggregates":          "#8e6ba8",   # violet, EA
}
FRACTION_SHORT = {
    "Cells Without Aggregates": "CWO",
    "Cells With Aggregates":    "CW",
    "Rest Cells":               "RC",
    "Pure Aggregates":          "EA",
}
PBS_GREY = "#999999"          # unseeded cells, in place of the fraction colour

# Colour of each of the ten conditions, for every panel that shows them all:
# hue = fraction (grey for PBS), lightness = BafA1. -BafA1 is the fraction
# colour, +BafA1 the same colour blended BAF_TINT of the way to white.
BAF_TINT = 0.5

def condition_color(cond):
    base = PBS_GREY if "PBS" in cond else next(c for f, c in FRACTION_COLORS.items() if f in cond)
    if "-BafA1" in cond:
        return base
    r, g, b = mcolors.to_rgb(base)
    return mcolors.to_hex((r + (1 - r) * BAF_TINT, g + (1 - g) * BAF_TINT, b + (1 - b) * BAF_TINT))

biological_color_dict = {c: condition_color(c) for c in biological_order}


def short_label(cond):
    """'-BafA1 PFF Pure Aggregates' -> 'PFF EA −Baf'."""
    seed = "PBS" if "PBS" in cond else "PFF"
    frac = next((s for f, s in FRACTION_SHORT.items() if f in cond), cond)
    baf  = "−Baf" if "-BafA1" in cond else "+Baf"
    return f"{seed} {frac} {baf}"

assert set(FRACTION_COLORS) == set(FRACTION_SHORT)


def two_level_xlabels(ax, positions, conditions, rotation=0, wrap_groups=False):
    """BafA1 state under each position, seeding and fraction in a bold row
    below, centred under its positions. The row's pad is the distance from the
    axes edge to the bottom of the BafA1 labels, plus 1 pt."""
    ax.set_xticks(positions)
    ax.set_xticklabels([short_label(c).rsplit(" ", 1)[1] for c in conditions],
                       rotation=rotation)
    groups = {}
    for x, c in zip(positions, conditions):
        groups.setdefault(short_label(c).rsplit(" ", 1)[0], []).append(x)
    fig = ax.figure
    fig.canvas.draw()
    sub_pt = (ax.get_window_extent().y0
              - min(t.get_window_extent().y0 for t in ax.get_xticklabels())) / fig.dpi * 72
    row = ax.secondary_xaxis("bottom")
    row.set_xticks([np.mean(xs) for xs in groups.values()],
                   labels=[g.replace(" ", "\n") if wrap_groups else g for g in groups],
                   fontweight="bold")
    row.tick_params(length=0, pad=sub_pt + native_pt(1.0))
    row.spines["bottom"].set_visible(False)      # the axes' own spine is drawn already
    return row

LEG_DOT = native_pt(3.5)   # legend markers, larger than the data dots

# --- Every observed condition must have a colour ---
# In a try block, so the cell also runs before the data are loaded. Every adata
# of this notebook has the same obs.
_DROPPED = {"Blank", "EvoTip Blank"}   # dropped in section 2; no colour expected
try:
    _levels = set(adata_phos_raw.obs[biological].dropna().astype(str)) - _DROPPED
    _unmapped = _levels - set(biological_color_dict)
    if _unmapped:
        print(f"WARNING: no colour for: {sorted(_unmapped)}")
except NameError:
    pass   # data not loaded yet

# 4. Phosphosites and protein groups per well (Figure S9 a, b)

In [ ]:
# Figure S9a, 2 x 1 units: phosphosites per well and condition, all sites (left
# axis) and Class I sites (right axis, hatched), with their means in two rows
# above the boxes. Counted on df_sites_raw_noblank, the 41 wells before the Class
# I mask and the completeness filter.

sample_cols = [c for c in df_sites_raw_noblank.columns if c not in META]
all_sites_per_sample = df_sites_raw_noblank[sample_cols].notna().sum(axis=0)

# Class I sites per well: localisation probability >= 0.75 in that run
classI_per_sample = (loc_per_run[sample_cols] >= 0.75).sum(axis=0)

# sample_id (phospho file name) -> well -> obs
sid_to_well = phos_site_wide_imp.set_index('sample_id')['well']
all_by_well    = all_sites_per_sample.rename(index=sid_to_well).reindex(adata_phos_raw.obs.index).values
classI_by_well = classI_per_sample.rename(index=sid_to_well).reindex(adata_phos_raw.obs.index).values

adata_phos_raw.obs['num_sites_all']    = all_by_well
adata_phos_raw.obs['num_sites_classI'] = classI_by_well

print(f"All sites: median={np.median(all_by_well):.0f}, "
      f"range=[{int(np.min(all_by_well))}, {int(np.max(all_by_well))}]")
print(f"Class I:   median={np.median(classI_by_well):.0f}, "
      f"range=[{int(np.min(classI_by_well))}, {int(np.max(classI_by_well))}]")

# --- twin-axis box plot ---
adata = adata_phos_raw

fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax_left = axm.next()
ax_right = ax_left.twinx()

# twinx() follows rcParams and hides the right spine, on which the Class I
# scale is drawn here; its left spine would be drawn over ax_left's.
ax_right.spines["right"].set_visible(True)
ax_right.spines["left"].set_visible(False)
ax_left.grid(axis="x", visible=False)        # would cut between every box pair

# conditions without wells are left out
conditions = [c for c in biological_order if (adata.obs[biological] == c).sum() > 0]
n_cond = len(conditions)
positions_left  = np.arange(n_cond) * 2
positions_right = np.arange(n_cond) * 2 + 0.8

all_data    = [adata.obs.loc[adata.obs[biological] == c, "num_sites_all"].values    for c in conditions]
classI_data = [adata.obs.loc[adata.obs[biological] == c, "num_sites_classI"].values for c in conditions]

S_DOT  = native_pt(2.2) ** 2     # sample dots, 2.2 pt across in print
JITTER = 0.15                    # largest horizontal offset of a dot, in x units

# All sites on the left axis, Class I on the right axis (hatched, diamond markers)
for ax, data, pos, hatch, marker, seed in (
        (ax_left,  all_data,    positions_left,  None,  "o", 42),
        (ax_right, classI_data, positions_right, "///", "D", 43)):
    bp = ax.boxplot(data, positions=pos, widths=0.6, patch_artist=True,
                    showfliers=False, zorder=2)
    for i, (patch, cond) in enumerate(zip(bp["boxes"], conditions)):
        color = biological_color_dict[cond]
        patch.set_facecolor(color)
        patch.set_edgecolor("black")
        if hatch:
            patch.set_hatch(hatch)
        jitter = np.random.default_rng(seed).uniform(-JITTER, JITTER, len(data[i]))
        ax.scatter(pos[i] + jitter, data[i], s=S_DOT, color=color, marker=marker,
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

# The two y axes share the ratio of the medians, with headroom above the highest box.
all_max    = max(np.max(d) for d in all_data)
classI_max = max(np.max(d) for d in classI_data)
all_median    = np.median(np.concatenate(all_data))
classI_median = np.median(np.concatenate(classI_data))
scale = classI_median / all_median

HEADROOM = 1.1
ax_left.set_ylim(bottom=0, top=all_max * HEADROOM)
ax_right.set_ylim(bottom=0, top=max(all_max * HEADROOM * scale, classI_max * HEADROOM))

X_PAD = 0.7
ax_left.set_xlim(positions_left[0] - X_PAD, positions_right[-1] + X_PAD)

# --- Means above the boxes ---
# All sites in the lower row (bold), Class I in the upper row (bold italic). The
# offsets are in points, so the two rows keep their spacing whatever height the
# axes get. The means print at 5 pt: a five-digit mean is 5.9 mm wide at 6 pt,
# and the conditions are 6.3 mm apart.
MEAN_FS = native_pt(5.0)
GAP = native_pt(1.5)           # axes top to the lower row
ROW = 1.25 * MEAN_FS           # lower row to upper row
top_left  = blended_transform_factory(ax_left.transData,  ax_left.transAxes)
top_right = blended_transform_factory(ax_right.transData, ax_right.transAxes)
for i in range(n_cond):
    ax_left.annotate(f"{np.mean(all_data[i]):.0f}", (positions_left[i], 1), xycoords=top_left,
                     xytext=(0, GAP), textcoords="offset points",
                     ha="center", va="bottom", fontsize=MEAN_FS, fontweight="bold")
    ax_right.annotate(f"{np.mean(classI_data[i]):.0f}", (positions_right[i], 1), xycoords=top_right,
                      xytext=(0, GAP + ROW), textcoords="offset points",
                      ha="center", va="bottom", fontsize=MEAN_FS, fontweight="bold",
                      style="italic")

# Two-level x labels, centred under each box pair (the Figure 4e convention of
# the 3_BrainTissue notebook).
centres = positions_left + 0.4
two_level_xlabels(ax_left, centres, conditions)

ax_left.set_ylabel("All phosphosites")
ax_right.set_ylabel("Class I phosphosites")   # one line; the legend gives loc ≥ 0.75

# --- Legend ---
# Lower left: the counts start far above zero and the PBS boxes on the left are
# the highest, so that corner is empty; at the lower right the legend covers the
# EA boxes. handlelength and handletextpad in font sizes keep the text close to
# the patch.
legend_handles = [
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, label="All phosphosites"),
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, hatch="///",
          label="Class I (loc ≥ 0.75)"),
]
_leg = ax_left.legend(handles=legend_handles, loc="lower left", frameon=False,
                      handlelength=1.0, handletextpad=0.4)

fig.canvas.draw()
assert ax_right.spines["right"].get_visible(), "the Class I axis lost its spine"
assert ax_right.get_yticklabels(), "the Class I axis has no tick labels"
# CHECK nothing drawn under the legend: the dots and boxes of both axes
_r = fig.canvas.get_renderer()
_lb = _leg.get_window_extent(_r)
_dots = sum(int(_lb.contains(*p)) for ax in (ax_left, ax_right) for _c in ax.collections
            for p in _c.get_offset_transform().transform(_c.get_offsets()))
_boxes = sum(_lb.overlaps(p.get_window_extent(_r)) for ax in (ax_left, ax_right)
             for p in ax.patches)
print(f"CHECK dots under the legend {_dots}, boxes under it {_boxes} (both must be 0)")

save_panel(fig, "a", "boxplot_phos_all_vs_classI", SI9_DIR)

In [ ]:
# Figure S9b, 2 x 1 units: protein groups per well and condition, from the
# proteome runs, with the mean above each box. Counted on prot_long, every protein
# group with a value in the well before the 70 % completeness filter, as Figure
# S9a counts the sites before their filters.
PG_SOURCE = prot_long
_pg_cols = [c for c in PG_SOURCE.columns if c not in ("well", "condition")]
adata_prot.obs["num_proteins"] = (PG_SOURCE.set_index("well")[_pg_cols].notna().sum(axis=1)
                                  .reindex(adata_prot.obs.index).values)
print(f"protein groups per sample: median {np.median(adata_prot.obs['num_proteins']):.0f}, "
      f"range [{adata_prot.obs['num_proteins'].min()}, {adata_prot.obs['num_proteins'].max()}] "
      f"of {len(_pg_cols)} in the matrix")

adata = adata_prot
conditions = [c for c in biological_order if (adata.obs[biological] == c).sum() > 0]
positions = np.arange(len(conditions)) * 1.4
pg_data = [adata.obs.loc[adata.obs[biological] == c, "num_proteins"].to_numpy() for c in conditions]

S_DOT  = native_pt(2.2) ** 2     # sample dots, 2.2 pt across in print
JITTER = 0.18                    # largest horizontal offset of a dot, in x units

fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax = axm.next()
ax.grid(axis="x", visible=False)        # would cut between the boxes

bp = ax.boxplot(pg_data, positions=positions, widths=0.7, patch_artist=True,
                showfliers=False, zorder=2)
for i, (patch, cond) in enumerate(zip(bp["boxes"], conditions)):
    color = biological_color_dict[cond]
    patch.set_facecolor(color)
    patch.set_edgecolor("black")
    jitter = np.random.default_rng(42).uniform(-JITTER, JITTER, len(pg_data[i]))
    ax.scatter(positions[i] + jitter, pg_data[i], s=S_DOT, color=color,
               edgecolors="black", linewidths=MARKER_LW, zorder=3)

ax.set_ylim(0, max(d.max() for d in pg_data) * 1.1)
ax.set_xlim(positions[0] - 0.8, positions[-1] + 0.8)

# Mean per condition above the axes, in one row: the conditions are 7 mm apart.
# Offsets in points, so the row keeps its place whatever height the axes get.
GAP = native_pt(1.5)
top = blended_transform_factory(ax.transData, ax.transAxes)
for x, d in zip(positions, pg_data):
    ax.annotate(f"{d.mean():.0f}", (x, 1), xycoords=top, xytext=(0, GAP),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=ANNOT_FS, fontweight="bold")

ax.set_ylabel("Protein groups")
two_level_xlabels(ax, positions, conditions)

save_panel(fig, "b", "boxplot_proteins", SI9_DIR)

# 5. Phosphosite rank abundance (Figure 6a)

In [ ]:
# Figure 6a, 1 x 1 unit: Class I phosphosites (adata_phos_raw: imputed, not
# normalised to the protein) ranked by their median intensity across all wells,
# with SNCA and p62/SQSTM1 sites called out. Same styling as the rank plots of the
# 1_HEK, 3_BrainTissue and 2_iDA notebooks.
S_BG = native_pt(0.8) ** 2      # background dots, 0.8 pt across in print
S_HL = native_pt(2.4) ** 2      # called-out sites, 2.4 pt across

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

adata = adata_phos_raw

# median of the log2 values, back to linear intensity, plotted on a log axis
median_intensity = 2 ** np.nanmedian(adata.X, axis=0)
adata.var["median_intensity"] = median_intensity
adata.var["site_rank"] = np.argsort(np.argsort(-adata.var["median_intensity"]))

at.pl.scatter(
    ax=ax, data=adata.var, x_column="site_rank", y_column="median_intensity",
    color=BG_COLOR, scatter_kwargs={"s": S_BG, "linewidths": 0},
)
ax.set_yscale("log")
# The exponent in a 10^n tick label is set at 70 % size and prints below 5 pt,
# so the ticks show the exponent itself and the axis label says log10.
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{np.log10(v):.0f}"))
# Ticks at the powers of ten only: the log scale adds minor ticks at 2-9 x each power.
ax.yaxis.set_minor_locator(NullLocator())

# ── Called-out sites: (gene, residue, position) ───────────────────────────
highlight_sites = [
    ("SNCA",   "S",  42),
    ("SNCA",   "S",  87),
    ("SNCA",   "Y", 125),
    ("SNCA",   "T",  81),
    ("SNCA",   "T",  92),
    ("SNCA",   "T",  64),
    ("SQSTM1", "S", 272),
    ("SQSTM1", "T", 269),
]

hl_set = set(highlight_sites)
hl_mask = [
    (g, a, int(p)) in hl_set
    for g, a, p in zip(adata.var["Genes_first"], adata.var["aa"], adata.var["position"])
]
highlight = adata.var[hl_mask].copy()

# Label: "GENE-AApos" (multiplicity dropped). SNCA-Y125 lies on the C-terminal
# tryptic peptide (residues 103-140) and is labelled by that region.
highlight["site_label"] = (
    highlight["Genes_first"] + "-"
    + highlight["aa"] + highlight["position"].astype(int).astype(str)
).replace({"SNCA-Y125": "SNCA-Cterminus"})

# One dot per site, not one per multiplicity. The same site quantified at
# multiplicity 1 and 2 is two rows carrying the same label, which would draw the
# label twice. Keep the most abundant, since abundance is the axis; the Brain
# notebook uses the same rule for duplicate protein groups.
_n_before = len(highlight)
highlight = (highlight.sort_values("median_intensity", ascending=False)
                      .drop_duplicates("site_label"))
if len(highlight) < _n_before:
    print(f"collapsed {_n_before - len(highlight)} extra multiplicities "
          f"(kept the highest median per site)")

# CHECK every requested site is drawn
found_keys = {(r["Genes_first"], r["aa"], int(r["position"])) for _, r in highlight.iterrows()}
not_found = [s for s in highlight_sites if s not in found_keys]
print(f"Highlight set: {len(highlight_sites)} requested, "
      f"{len(found_keys)} found, {len(not_found)} missing")
if not_found:
    print("Not detected in Class I phospho data:")
    for g, a, p in not_found:
        print(f"  {g}-{a}{p}")
print(f"Sites drawn: {len(highlight)}")

ax.scatter(
    highlight["site_rank"], highlight["median_intensity"],
    color=HIGHLIGHT_COLOR, edgecolors="black", s=S_HL,
    zorder=5, linewidths=MARKER_LW,
)

label_axes(
    ax,
    xlabel="Phosphosite rank",
    ylabel="log10 median intensity",
)
# The axis labels exist before the site labels are placed: constrained layout
# sizes the axes at the next draw, and the placement needs the final axes.
fig.canvas.draw()

# The labels in LABEL_BELOW are placed below the curve, in the empty part of the panel
# under it, at fixed offsets in printed points: adjust_text cannot be told which
# side of a dot to use, and above the curve these labels and the leader lines of
# their neighbours cross. Offset = from the dot to the label's upper corner on
# the side given; the label is aligned to that side. They are obstacles for the
# other labels.
LABEL_BELOW = {
    "SQSTM1-T269": (native_pt(-21.0), native_pt(-14.0), "left"),
    "SNCA-T64":    (native_pt(0.0),   native_pt(-4.5),  "right"),
}
SHRINK_DOT = native_pt(1.6)          # the leader stops outside the dot
_missing = set(LABEL_BELOW) - set(highlight["site_label"])
if _missing:
    print(f"LABEL_BELOW names that are not drawn: {sorted(_missing)}")
below = [
    ax.annotate(row["site_label"], (row["site_rank"], row["median_intensity"]),
                xytext=LABEL_BELOW[row["site_label"]][:2], textcoords="offset points",
                ha=LABEL_BELOW[row["site_label"]][2], va="top", fontsize=ANNOT_FS,
                zorder=6, arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                          shrinkA=0, shrinkB=SHRINK_DOT))
    for _, row in highlight[highlight["site_label"].isin(LABEL_BELOW)].iterrows()
]

# The other labels are placed in empty space by adjust_text, with every 5th
# point of the curve as an obstacle (the curve is dense enough for that).
texts = [
    ax.text(row["site_rank"], row["median_intensity"], row["site_label"],
            fontsize=ANNOT_FS, ha="left", va="center")
    for _, row in highlight[~highlight["site_label"].isin(LABEL_BELOW)].iterrows()
]

all_x = adata.var["site_rank"].values[::5]
all_y = adata.var["median_intensity"].values[::5]

adjust_text(
    texts,
    x=all_x, y=all_y,
    objects=below,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    expand=(1.5, 1.0),
    force_static=(1.0, 1.5),
    force_text=(0.5, 1.0),
    force_explode=(0.5, 0.8),
)

# CHECK the labels placed below the curve: inside the axes, and no curve point
# under them
fig.canvas.draw()
_ab = ax.get_window_extent()
_curve = ax.transData.transform(np.column_stack([adata.var["site_rank"], adata.var["median_intensity"]]))
for _t in below:
    # the text alone: an annotation's own extent includes its leader line
    _bb = super(type(_t), _t).get_window_extent()
    _inside = _bb.x0 >= _ab.x0 and _bb.x1 <= _ab.x1 and _bb.y0 >= _ab.y0
    _hits = int(sum(_bb.contains(*p) for p in _curve))
    print(f"CHECK {_t.get_text()}: inside the axes {_inside}, curve points under it {_hits}")

save_panel(fig, "a", "phos_rank_abundance", FIG6_DIR)

# 6. Coefficients of variation (Figure S9 c, d)

In [ ]:
# Figure S9c (phosphosites) and S9d (protein groups), 1 x 1 unit each: CV within
# each condition on the observed, linear intensities (phos_site_wide_obs, the
# Class I sites before imputation; prot_filt), from features with >= 3 measured
# replicates in the condition. The number above each violin is the mean CV.
cv_configs = [
    (adata_phos_raw,  phos_site_wide_obs, [c for c in phos_site_wide_obs.columns if '~' in str(c)],                  "phos_raw", "phosphosite", "c"),
    (adata_prot,      prot_filt, [c for c in prot_filt.columns if c not in ('well', 'condition')], "prot",     "protein",     "d"),
]

for adata, pre_df, feature_cols, suffix, feature_name, letter in cv_configs:
    conditions = [c for c in biological_order if (adata.obs[biological] == c).sum() > 0]
    n_cond = len(conditions)
    positions = np.arange(n_cond) * 1.4

    pre_by_well = pre_df.set_index('well')[feature_cols]

    # --- CV per feature and condition, linear intensity, >= 3 measured replicates ---
    cv_arrays = []
    for c in conditions:
        wells = adata.obs.index[adata.obs[biological] == c].tolist()
        X_pre = pre_by_well.loc[wells].values.astype(float)   # rows=samples, cols=features, log2
        n_valid = np.sum(~np.isnan(X_pre), axis=0)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            X_linear = 2 ** X_pre
            mean_vals = np.nanmean(X_linear, axis=0)
            std_vals  = np.nanstd(X_linear, axis=0, ddof=1)
            cvs = std_vals / mean_vals
            cvs[n_valid < 3] = np.nan
        cv_arrays.append(cvs[np.isfinite(cvs)])

    # --- Plot ---
    fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
    ax = axm.next()
    ax.grid(False)        # the 20 % reference line would be drawn on top of a grid line

    # Drawn as the CV panel of the 1_HEK notebook: the condition colour at full
    # strength, black outline, the median and the range as black lines.
    parts = ax.violinplot(cv_arrays, positions=positions, widths=1.0, showmedians=True)
    for body, cond in zip(parts["bodies"], conditions):
        body.set_facecolor(biological_color_dict[cond])
        body.set_edgecolor("black")
        body.set_linewidth(MARKER_LW)
        body.set_alpha(1.0)
    for _k in ("cmedians", "cmins", "cmaxes", "cbars"):
        parts[_k].set(color="black", linewidth=MARKER_LW)

    # y axis up to 1.15 x the 99th percentile; the long upper tails are cut
    all_cv = np.concatenate(cv_arrays)
    ax.set_ylim(0, np.quantile(all_cv, 0.99) * 1.15)

    # 20 % reference line
    ax.axhline(y=0.2, color=REF_COLOR, linestyle="--", linewidth=AXES_LW, zorder=0)

    # Mean CV per condition above the violins, every second one in a second row:
    # in one row, neighbouring means (3.4 mm wide at 6 pt) are 3 mm apart.
    # Offsets in points, so the rows keep their spacing whatever height the axes get.
    GAP = native_pt(1.5)           # axes top to the lower row
    ROW = 1.25 * ANNOT_FS          # lower row to upper row
    top = blended_transform_factory(ax.transData, ax.transAxes)
    for i, (p, arr) in enumerate(zip(positions, cv_arrays)):
        ax.annotate(f"{np.mean(arr):.2f}", (p, 1), xycoords=top,
                    xytext=(0, GAP + (i % 2) * ROW), textcoords="offset points",
                    ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")

    # Two-level x labels, the BafA1 state rotated (the violins are 3 mm apart).
    # One rotated label per condition ("PBS CWO −Baf") takes 15 mm of the height.
    ax.set_xlim(positions[0] - 0.8, positions[-1] + 0.8)
    label_axes(ax, xlabel="", ylabel=f"CV per {feature_name}")
    # Two lines in the group row: the group is 6 mm wide, "PFF CWO" in bold 9 mm.
    two_level_xlabels(ax, positions, conditions, rotation=90, wrap_groups=True)

    save_panel(fig, letter, f"cv_{suffix}", SI9_DIR)

# 7. PCA

## 7.1 Class I phosphosites (Figure 6b)

In [ ]:
# Figure 6b, 1 x 1 unit: PCA of the Class I sites (adata_phos_raw: imputed, not
# normalised to the protein) that have a value in every well. Drawn as Figure 2c
# of the 1_HEK notebook: colour = fraction (grey for PBS), lightness = BafA1,
# both from biological_color_dict of the style cell; every dot filled, with a
# black edge.
adata, suffix = adata_phos_raw, "phos_raw"

DOT = native_pt(2.8)    # marker diameter, 2.8 pt in print, as in Figure 2c

if "X_pca_obs" not in adata.obsm:
    adata.var["_pca_eligible"] = ~np.isnan(adata.X).any(axis=0)
    n_eligible = int(adata.var["_pca_eligible"].sum())
    print(f"{suffix}: PCA on {n_eligible}/{adata.n_vars} features "
          f"({100*n_eligible/adata.n_vars:.1f}% — NaN-free subset)")
    at.tl.pca(adata, dim_space="obs", meta_data_mask_column_name="_pca_eligible")

pc1 = adata.obsm["X_pca_obs"][:, 0]
pc2 = adata.obsm["X_pca_obs"][:, 1]
var_explained = adata.uns["variance_pca_obs"]["variance_ratio"]

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.set_axisbelow(True)   # the grid goes below the dots

for condition in biological_order:
    mask = (adata.obs[biological] == condition).to_numpy()
    if mask.any():
        ax.scatter(pc1[mask], pc2[mask], s=DOT ** 2, color=biological_color_dict[condition],
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

label_axes(ax, xlabel=f"PC1 ({var_explained[0] * 100:.1f}%)",
               ylabel=f"PC2 ({var_explained[1] * 100:.1f}%)")

# Direct labels for the two groups of the panel's claim, both at the height of
# the highest point so that they line up with each other.
_y0, _y1 = ax.get_ylim()
_ea_mask = adata.obs[biological].str.contains("Pure Aggregates").to_numpy()
_group_labels = [
    ax.annotate(_txt, (pc1[_m].mean(), pc2.max()), xytext=(0, native_pt(1.5)),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=ANNOT_FS, fontweight="bold", linespacing=1.2, color="0.3")
    for _m, _txt in ((_ea_mask, "Excised\naggregates"), (~_ea_mask, "Cellular\nfractions"))
    if _m.any()]

# CHECK the two direct labels only make sense if the groups separate on PC1
if _ea_mask.any() and (~_ea_mask).any():
    _sep   = abs(pc1[_ea_mask].mean() - pc1[~_ea_mask].mean())
    _range = pc1.max() - pc1.min()
    print(f"{suffix}: super-group PC1 means differ by {_sep:.1f} "
          f"({100*_sep/_range:.0f}% of the PC1 range)")
    if _sep < 0.25 * _range:
        print("  NOTE the two labels will sit close together — EA does not "
              "separate from the cellular fractions on PC1 in this matrix. "
              "Delete the annotate loop; moving the labels apart would misstate the panel.")

# Legend: one entry per fraction, its -BafA1 and +BafA1 colours side by side,
# in the order CWO -> RC -> CW -> EA, then PBS. Markers 4.5 pt: at the 3.5 pt of
# LEG_DOT, next to the 2.8 pt dots of this panel, the legend markers cannot be
# told apart from the data dots.
LEG_DOT_PCA = native_pt(4.5)
_order = [("PFF", "Cells Without Aggregates"), ("PFF", "Rest Cells"),
          ("PFF", "Cells With Aggregates"), ("PFF", "Pure Aggregates"),
          ("PBS", "Cells Without Aggregates")]
_handles, _labels = [], []
for seed, frac in _order:
    _handles.append(tuple(
        Line2D([], [], ls="none", marker="o", markersize=LEG_DOT_PCA, markeredgewidth=MARKER_LW,
               markeredgecolor="black",
               markerfacecolor=biological_color_dict[f"{b}BafA1 {seed} {frac}"])
        for b in "-+"))
    _labels.append("PBS" if seed == "PBS" else FRACTION_SHORT[frac])

# The axes top is raised until the two labels are inside the axes; the y ticks
# stay within the data, so they mark the range of the data.
for _ in range(4):
    fig.canvas.draw()
    _ab = ax.get_window_extent()
    _over = max(t.get_window_extent().y1 for t in _group_labels) - (_ab.y1 - native_pt(1.0))
    if _over <= 0:
        break
    _lo, _hi = ax.get_ylim()
    ax.set_ylim(_lo, _hi + _over / _ab.height * (_hi - _lo) * 1.1)
_lo, _hi = ax.get_ylim()
ax.set_yticks([t for t in ax.get_yticks() if _y0 <= t <= _y1])
ax.set_ylim(_lo, _hi)

# Legend inside the axes, where no dot is: first the bottom of the empty band
# between the excised aggregates and the cellular fractions, then the corners.
# The first place with no dot under the legend is kept.
_x_gap  = (pc1[_ea_mask].min() + pc1[~_ea_mask].max()) / 2 if _ea_mask.any() else 0.0
_x_frac = ax.transAxes.inverted().transform(ax.transData.transform((_x_gap, 0)))[0]
_places = [("lower center", (_x_frac, 0.0)), ("lower right", None), ("lower left", None),
           ("upper right", None), ("upper left", None)]
_pts = ax.transData.transform(np.column_stack([pc1, pc2]))
for _loc, _anchor in _places:
    _leg = ax.legend(_handles, _labels, handler_map={tuple: HandlerTuple(ndivide=None)},
                     title="−/+ BafA1", loc=_loc, bbox_to_anchor=_anchor,
                     frameon=True, edgecolor="none", framealpha=1,   # white ground hides the grid
                     handlelength=1.8, handletextpad=0.3, borderaxespad=0.3)
    # handlelength 1.8 font sizes: the two markers of an entry (4.9 pt across with
    # their edge) stand 5.4 pt apart, a 0.5 pt gap; the legend pad is 0 (figure_style).
    fig.canvas.draw()
    _lb = _leg.get_window_extent()
    _n_under = int(sum(_lb.contains(*p) for p in _pts))
    _n_text = sum(_lb.overlaps(t.get_window_extent()) for t in _group_labels)
    if _n_under == 0 and _n_text == 0:
        break
print(f"CHECK legend at {_loc!r}: dots under it {_n_under}, direct labels under it "
      f"{_n_text} (both must be 0)")

save_panel(fig, "b", "pca_phos_raw", FIG6_DIR)

## 7.2 Coloured by SNCA phosphorylation (Figure S9e)

In [ ]:
# Figure S9e, 1 x 1 unit: the PCA of Figure 6b, each well coloured by the summed
# signal of all its SNCA phosphosites (linear sum, back to log2; colour scale
# clipped to the 5th-95th percentile). Needs the PCA of the cell above.
adata = adata_phos_raw

DOT = native_pt(2.0)   # sample dots, 2.0 pt across in print

# all SNCA phosphosites
snca_mask = (adata.var["Genes_first"] == "SNCA").values
snca_sites = adata.var.index[snca_mask].tolist()

# A magnitude with no sign, in the blue ramp of the detection heatmaps (white
# through the CWO blue to a darker blue). DIRECTION_CMAP is reserved for
# direction.
cmap = LinearSegmentedColormap.from_list(
    "snca_signal", ["#ffffff", FRACTION_COLORS["Cells Without Aggregates"], "#31688e"])

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

if snca_sites:
    # sum in linear space, back to log2
    X_snca     = adata.X[:, snca_mask]                      # samples × snca_sites, log2
    linear_sum = np.nansum(2 ** X_snca, axis=1)
    linear_sum = np.where(linear_sum > 0, linear_sum, np.nan)  # wells with no SNCA site
    log2_sum   = np.log2(linear_sum)

    lo, hi = np.nanpercentile(log2_sum, [5, 95])
    adata.obs["_snca_color"] = np.clip(log2_sum, lo, hi)

    at.pl.plot_pca(
        data=adata, ax=ax, x_column=1, y_column=2, label=False,
        color_map_column="_snca_color", palette=cmap,
        scatter_kwargs={"s": DOT ** 2, "alpha": 0.8, "linewidth": MARKER_LW,
                        "edgecolor": "black"},
    )
    # plot_pca draws the dots at zorder 1, below the grid (1.5), so grid lines
    # cross every dot; the grid goes below the data.
    ax.set_axisbelow(True)
    var_explained = adata.uns["variance_pca_obs"]["variance_ratio"]
    ax.set_xlabel(f"PC1 ({var_explained[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_explained[1]*100:.1f}%)")

    norm = mcolors.Normalize(vmin=lo, vmax=hi)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02, extend="both")
    # One line; "summed over the SNCA phosphosites" goes into the figure legend.
    cbar.set_label("log2 Σ SNCA phospho")

    print(f"PCA by Σ SNCA phosphosite signal; SNCA phosphosites included ({len(snca_sites)}):")
    for s in snca_sites:
        v = adata.var.loc[s]
        print(f"  {v['aa']}{int(v['position'])} M{int(v['mult'])}  ({s})")
else:
    ax.text(0.5, 0.5, "No SNCA sites found", ha="center", va="center",
            transform=ax.transAxes, fontsize=ANNOT_FS)

save_panel(fig, "e", "pca_snca_phos_raw", SI9_DIR)

# 8. Differential phosphorylation

Moderated t-test with empirical Bayes variance moderation (limma, alphapepttools `diff_exp_ebayes`) for 19 contrasts, on the Class I sites normalised to their parent protein. FDR by Benjamini-Hochberg within each contrast.

In [ ]:
# The 19 contrasts, name -> (first group, second group); log2FC = first - second.
comparisons = {
    # --- BafA1 effect (within same sample type) ---
    "BafA1_effect_PBS": (
        "+BafA1 PBS Cells Without Aggregates",
        "-BafA1 PBS Cells Without Aggregates",
    ),
    "BafA1_effect_PFF_CellsWithout": (
        "+BafA1 PFF Cells Without Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "BafA1_effect_PFF_CellsWith": (
        "+BafA1 PFF Cells With Aggregates",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "BafA1_effect_PFF_PureAggs": (
        "+BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Pure Aggregates",
    ),
    "BafA1_effect_PFF_RestCells": (
        "+BafA1 PFF Rest Cells",
        "-BafA1 PFF Rest Cells",
    ),
    # --- PFF vs PBS (within same inhibitor) ---
    "PFF_vs_PBS_minusBafA1": (
        "-BafA1 PFF Cells Without Aggregates",
        "-BafA1 PBS Cells Without Aggregates",
    ),
    "PFF_vs_PBS_plusBafA1": (
        "+BafA1 PFF Cells Without Aggregates",
        "+BafA1 PBS Cells Without Aggregates",
    ),
    # --- Aggregate vs non-aggregate ---
    "PureAggs_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "PureAggs_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    "CellsWith_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Cells With Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "CellsWith_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Cells With Aggregates",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    # --- Pure Aggs vs Cells With ---
    "PureAggs_vs_CellsWith_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "PureAggs_vs_CellsWith_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWith_minusBafA1": (
        "-BafA1 PFF Rest Cells",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWith_plusBafA1": (
        "+BafA1 PFF Rest Cells",
        "+BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Rest Cells",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "RestCells_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Rest Cells",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    "PureAggs_vs_RestCells_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Rest Cells",
    ),
    "PureAggs_vs_RestCells_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Rest Cells",
    ),
}

In [ ]:
# Every contrast in four modes, one CSV per mode and contrast in output_dir:
#   full_imputed   all sites, imputed matrix (adata_phos_site_imp_norm)
#   AND_filter     sites with enough measured values in both groups (2.7);
#                  the reported results (Figure 6c, S9f)
#   OR_filter      sites with enough measured values in at least one group
#   non_imputed    observed values only (adata_phos_site_obs_norm)
# The last three are sensitivity analyses.
MODES_TO_RUN = ["full_imputed", "AND_filter", "OR_filter", "non_imputed"]
FDR_CUTOFF = 0.05
LFC_CUTOFF = 0.5

adata_imp, adata_obs, pre_wide = adata_phos_site_imp_norm, adata_phos_site_obs_norm, phos_site_wide_obs
print(f"imp: {adata_imp.shape}, obs: {adata_obs.shape}, pre_wide: {pre_wide.shape}")

# condition labels patsy accepts
bio_safe = biological + "_safe"
for a in (adata_imp, adata_obs):
    a.obs[bio_safe] = a.obs[biological].astype(str).map(_sanitize)

comparisons_safe = {
    name: (_sanitize(g1), _sanitize(g2))
    for name, (g1, g2) in comparisons.items()
}

annot_cols = ["Genes_first", "aa", "position", "mult",
              "Protein.Group_first", "has_proteome_match"]
def _make_label(row):
    return (f"{row['Genes_first']}-"
            f"{row['aa']}{int(row['position'])} "
            f"M{int(row['mult'])}")

ebayes_all = {}   # {mode: {contrast_name: result_df}}

for mode in MODES_TO_RUN:
    print(f"\n{'='*70}\n{mode}\n{'='*70}")
    results_for_mode = {}

    for name, (g1_safe, g2_safe) in comparisons_safe.items():
        g1_orig, g2_orig = comparisons[name]

        # ---- the sites and values of this mode ----
        if mode == "full_imputed":
            ad_use = adata_imp

        elif mode == "non_imputed":
            ad_use = adata_obs

        elif mode in ("AND_filter", "OR_filter"):
            qualifying = sites_with_enough_values(
                g1_orig, g2_orig, pre_wide, adata_imp,
                require="both" if mode == "AND_filter" else "either",
                biological_col=biological)
            sites_in_adata = [s for s in qualifying if s in adata_imp.var_names]
            if len(sites_in_adata) < 10:
                print(f"  {name}: SKIPPED (only {len(sites_in_adata)} qualifying sites)")
                continue
            ad_use = adata_imp[:, sites_in_adata].copy()
            ad_use.obs[bio_safe] = ad_use.obs[biological].astype(str).map(_sanitize)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        # ---- eBayes ----
        try:
            _, result = at.tl.diff_exp_ebayes(
                adata=ad_use,
                between_column=bio_safe,
                comparison=(g1_safe, g2_safe),
            )
        except Exception as e:
            print(f"  {name}: FAILED — {e}")
            continue

        # ---- annotate ----
        result = result.merge(
            ad_use.var[annot_cols],
            left_on="protein", right_index=True, how="left",
        )
        result["site"]   = result.apply(_make_label, axis=1)
        result["group1"] = g1_orig
        result["group2"] = g2_orig
        result["mode"]   = mode
        result["layer"]  = "site"     # a column of the CSVs: Class I site level

        # ---- OR_filter: in which group(s) the site has enough measured values ----
        if mode == "OR_filter":
            g1_wells = adata_imp.obs.index[adata_imp.obs[biological] == g1_orig].tolist()
            g2_wells = adata_imp.obs.index[adata_imp.obs[biological] == g2_orig].tolist()
            min_g1, min_g2 = _min_real(len(g1_wells)), _min_real(len(g2_wells))
            n_g1 = pre_wide.set_index("well").loc[g1_wells, result["protein"]].notna().sum(axis=0).values
            n_g2 = pre_wide.set_index("well").loc[g2_wells, result["protein"]].notna().sum(axis=0).values
            result["n_real_g1"] = n_g1
            result["n_real_g2"] = n_g2
            result["detection_mode"] = np.where(
                (n_g1 >= min_g1) & (n_g2 >= min_g2), "both",
                np.where(n_g1 >= min_g1, "g1_only", "g2_only"),
            )

        # column order: identifiers, detection columns, statistics
        id_cols  = ["protein", "site", "group1", "group2", "mode", "layer"] + annot_cols
        extra    = [c for c in ("n_real_g1", "n_real_g2", "detection_mode") if c in result.columns]
        stat_cols = [c for c in result.columns if c not in id_cols + extra]
        result = result[id_cols + extra + stat_cols]

        results_for_mode[name] = result

        n_sig  = (result["fdr"] < FDR_CUTOFF).sum()
        n_up   = ((result["fdr"] < FDR_CUTOFF) & (result["log2fc"] >  LFC_CUTOFF)).sum()
        n_down = ((result["fdr"] < FDR_CUTOFF) & (result["log2fc"] < -LFC_CUTOFF)).sum()
        print(f"  {name}: {len(result):>5} tested, "
              f"{n_sig:>4} sig (FDR<{FDR_CUTOFF}), {n_up:>4} up, {n_down:>4} down")

        result.to_csv(
            f"{output_dir}/{today}_ebayes_site_{mode}_{name}.csv",
            index=False,
        )

    ebayes_all[mode] = results_for_mode

print(f"\nDone. Modes run: {list(ebayes_all)}")
print(f"CSVs in: {output_dir}")

# 9. Volcano plots, EA against RC (Figure 6c, Figure S9f)

In [ ]:
def _nice_title(contrast_name):
    """'PureAggs_vs_RestCells_minusBafA1' → 'Excised Aggregates vs Rest Cells  (−BafA1)',
    for the printed summaries."""
    nice_groups = {
        "PureAggs":     "Excised Aggregates",   # EA, display-only rename
        "RestCells":    "Rest Cells",
        "CellsWith":    "Cells With Aggregates",
        "CellsWithout": "Cells Without Aggregates",
        "PFF":          "PFF",
        "PBS":          "PBS",
    }
    parts = contrast_name.split("_")
    baf = ""
    if parts and parts[-1] == "minusBafA1":
        baf, parts = "−BafA1", parts[:-1]
    elif parts and parts[-1] == "plusBafA1":
        baf, parts = "+BafA1", parts[:-1]
    if "vs" in parts:
        idx = parts.index("vs")
        g1 = "_".join(parts[:idx])
        g2 = "_".join(parts[idx + 1:])
        title = f"{nice_groups.get(g1, g1)} vs {nice_groups.get(g2, g2)}"
        if baf:
            title += f"  ({baf})"
        return title
    return contrast_name.replace("_", " ")

In [ ]:
# Functional groups of the volcano plots: hand-curated gene lists, their colours,
# and the genes named on the panel.
#
# The colours follow the rule of the compartment panel of the 1_HEK notebook
# (8.2): greens, yellows, browns and greys, the hues that are not already a
# fraction, a direction or the call-out colour. Four groups have a compartment
# with the same meaning there and take its colour (Nucleus, Cytoplasm /
# cytoskeleton, Plasma membrane, Endosome / lysosome); "signalling / other" takes
# the olive of that palette, the one furthest in lightness from the other four.
# A panel that colours by function cannot also colour by direction: there,
# direction is carried by the x axis alone, and the legend has to say so.
# Names are short because they are legend entries.
PHOSPHO_GROUP_COLORS = {
    "nuclear / splicing":  "#1b7837",   # dark green, Autophagy "Nucleus"
    "degradation":         "#666666",   # dark grey,  Autophagy "Endosome / lysosome"
    "cytoskeleton":        "#e6ab02",   # mustard,    Autophagy "Cytoplasm / cytoskeleton"
    "membrane":            "#dfc27d",   # sand,       Autophagy "Plasma membrane"
    "signalling / other":  "#80803a",   # olive
}

# Every site of a listed gene is coloured with its group.
GROUPS = {
    "nuclear / splicing": [
        "RBM39","THRAP3","IRF2BP2","SRRM1","SNRNP70","SRSF11","SRRM2","RBM17","RBM15",
        "ZRANB2","GPALPP1","TRA2A","SF3A1","RBM8A","NPM1","MATR3","DDX46","BCLAF1",
        "XRN2","SCAF1","PNN","NONO","NCL","SRRT","HNRNPC","MTA1"],
    "degradation": [
        "DCAF8","NACA","PCNP","SQSTM1","UBE2J1","HSP90AB1","RAD23B"],
    "cytoskeleton": [
        "CTTN","MYH9","MYH10","VIM","MAP7D1","MAP1B","MAP4","SEPTIN2","FLNC"],
    "membrane": [
        "TOMM22","PGRMC2","CANX","SCAP","MARCKSL1","EPS15L1","MARCKS"],
    "signalling / other": [
        "SNCA","IRS2","HDGF","MAX","MCM2","SSRP1","PDAP1","CHAMP1","LMNB1","CLNS1A",
        "EEF1D","HDAC2","TMPO","RPLP2","BAZ1A","TRAFD1","S100A9","PPP1R12A","EEF1B2",
        "PARD3","YBX3","PDCD5"],
}

# Genes named on the panel (most significant site per gene).
LABEL = [
    "RBM39",    # nuclear scaffolds / splicing, representative
    "THRAP3",   # extreme point of the -BafA1 panel
    "SQSTM1",   # degradation / chaperoning (p62)
    "VIM",      # cytoskeleton (aggresome cage)
    "TOMM22",   # membrane / organelle
    "SNCA",     # alpha-synuclein
    "ZC3H13",   # the only reproducible decrease
]

# Display-name overrides, keyed by (gene, aa, position). SNCA Y133 is the
# search engine's assignment; the peptide reports the C-terminal region.
SITE_LABELS = {
    ("SNCA", "Y", 133): "SNCA - C-term",
}


def _norm_gene(s):
    """First symbol of a protein group, whitespace stripped."""
    return str(s).split(";")[0].strip()


# --- gene -> group; CHECK that no gene is in two groups ---
_seen, _dupes = {}, {}
for _g, _genes in GROUPS.items():
    for _gene in _genes:
        _k = _norm_gene(_gene)
        if _k in _seen:
            _dupes.setdefault(_k, [_seen[_k]]).append(_g)
        _seen[_k] = _g
assert not _dupes, f"genes assigned to more than one group: {_dupes}"
GENE_TO_GROUP = _seen
LABEL_NORM = [_norm_gene(g) for g in LABEL]
_orphans = sorted(set(LABEL_NORM) - set(GENE_TO_GROUP))
print(f"{len(GENE_TO_GROUP)} genes across {len(GROUPS)} groups; "
      f"{len(_orphans)} LABEL genes in no group: {_orphans}")
assert set(GROUPS) <= set(PHOSPHO_GROUP_COLORS), (
    f"no colour for group(s): {sorted(set(GROUPS) - set(PHOSPHO_GROUP_COLORS))}")

In [ ]:
# Figure 6c (−BafA1) and Figure S9f (+BafA1), 1 x 1 unit each: volcano plots of
# EA against RC, AND_filter results of section 8 (eBayes on the sites with
# >= 60 % and >= 2 measured values in both groups; Class I sites, imputed,
# normalised to the condition mean of the parent protein). Dashed lines at
# |log2FC| = LFC_THRESH and FDR = FDR_THRESH. Colour = functional group of the
# cell above, not direction; direction is the x axis. Only LABEL genes are named,
# in anchored columns in the side gutters.

# --- Label geometry ------------------------------------------------------
# Labels go into columns in gutters on either side of the data, not next to the
# points. GUTTER_* widen the x limits to make the gutters; ANCHOR_* place the
# column just outside the data, so each label's line is drawn inward to its dot.
GUTTER_L, GUTTER_R = 0.55, 0.65   # fractions of the data x range; at one unit,
                                  # 0.34 / 0.44 put the 6 pt labels past the axes
ANCHOR_L, ANCHOR_R = 0.04, 0.06
LAB_START_R = 0.90   # column start height, as a fraction of the axes height
LAB_START_L = 0.55   # the left column starts lower, next to its point
LAB_PAD     = 1.8    # label spacing = LAB_PAD x the 6 pt tick size in pixels,
                     # 1.3 line heights at the figure's 100 dpi

# contrast -> (panel letter, folder, x label). There are no panel titles; the x
# label names the two groups, the direction (positive = higher in EA) and BafA1.
contrasts_to_plot = {
    "PureAggs_vs_RestCells_minusBafA1": ("c", FIG6_DIR, "log2 FC (EA / RC, −BafA1)"),
    "PureAggs_vs_RestCells_plusBafA1":  ("f", SI9_DIR,  "log2 FC (EA / RC, +BafA1)"),
}
LFC_THRESH = 0.5
FDR_THRESH = 0.05
S_BG  = native_pt(1.0) ** 2   # background sites, 1.0 pt across in print
S_GRP = native_pt(2.0) ** 2   # curated sites, 2.0 pt across
RESULTS_DICT = ebayes_all["AND_filter"]
FILTER_LABEL = "AND_filter"

volcano_counts = {}        # contrast -> enriched / depleted / tested


def _label_column(ax, sub, anchor, start, label_kwargs, line_kwargs):
    """One column of labels at x = anchor, drawn by label_plot.

    label_plot puts the first label at y_display_start x the axes height in
    pixels, counted from the bottom of the FIGURE. `start` is a fraction of the
    axes height and is converted, so the column starts where it says.
    """
    _bb = ax.get_window_extent()
    label_plot(ax, sub, x_column="log2fc", y_column="-log10(fdr)", label_column="label",
               x_anchors=[anchor], y_display_start=start + _bb.y0 / _bb.height,
               y_padding_factor=LAB_PAD, label_kwargs=label_kwargs,
               line_kwargs=line_kwargs)


for name in contrasts_to_plot:
    if name not in RESULTS_DICT:
        print(f"Skipping {name} — not in {FILTER_LABEL} results")
        continue
    result = RESULTS_DICT[name].copy()
    result["gene_raw"] = result["Genes_first"].astype(str)
    result["gene"] = result["gene_raw"].map(_norm_gene)
    result["group"] = result["gene"].map(GENE_TO_GROUP)
    result["site_short"] = (
        result["gene"] + "-" + result["aa"].astype(str)
        + result["position"].astype(int).astype(str)
    )

    # === Sites that move: FDR < FDR_THRESH and |log2FC| > LFC_THRESH ==========
    # Positive log2FC is higher in EA, so "enriched" means enriched in the
    # excised aggregates relative to the rest cells.
    _sig = result["fdr"] < FDR_THRESH
    _up  = _sig & (result["log2fc"] >  LFC_THRESH)
    _dn  = _sig & (result["log2fc"] < -LFC_THRESH)
    _hdr = _nice_title(name)
    print("\n" + "=" * 72)
    print(f"  {_hdr}")
    print(f"    ENRICHED in EA : {int(_up.sum()):5d} sites")
    print(f"    DEPLETED in EA : {int(_dn.sum()):5d} sites")
    print(f"    tested         : {len(result):5d} sites "
          f"(FDR < {FDR_THRESH}, |log2FC| > {LFC_THRESH})")
    print("=" * 72)
    volcano_counts[name] = dict(title=_hdr, up=int(_up.sum()),
                                down=int(_dn.sum()), tested=len(result))

    _in_group = result["group"].notna()
    _genes_found = result.loc[_in_group, "gene"].nunique()
    print(f"\n{name}: {int(_in_group.sum())} coloured sites from {_genes_found} "
          f"of {len(GENE_TO_GROUP)} listed genes "
          f"({len(GENE_TO_GROUP) - _genes_found} genes not tested here)")

    # CHECK every LABEL gene: was it found, and did it get a colour?
    print("  LABEL gene status (raw Genes_first -> group):")
    for _g in LABEL_NORM:
        _hit = result[result["gene"] == _g]
        if _hit.empty:
            print(f"    {_g:<10} NOT TESTED in this contrast")
            continue
        _raw = _hit["gene_raw"].iloc[0]
        _grp = _hit["group"].iloc[0]
        _note = "" if pd.notna(_grp) else "   <- grey: add it to a GROUPS list"
        print(f"    {_g:<10} {_raw!r:<22} -> {_grp if pd.notna(_grp) else 'none'}{_note}")

    letter, folder, xlabel = contrasts_to_plot[name]
    fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
    ax = axm.next()
    ax.spines[["top", "right"]].set_visible(True)     # boxed, as the Autophagy volcano

    # Background: every tested site, one grey cloud.
    ax.scatter(result["log2fc"], result["-log10(fdr)"],
               s=S_BG, c=BG_COLOR, alpha=0.5, linewidth=0, zorder=2)

    # Curated sites, one scatter call per group so the legend is exact.
    for _grp in GROUPS:                      # dict order = legend order
        _sub = result[result["group"] == _grp]
        if _sub.empty:
            continue
        ax.scatter(_sub["log2fc"], _sub["-log10(fdr)"],
                   s=S_GRP, c=PHOSPHO_GROUP_COLORS[_grp],
                   edgecolor="black", linewidth=MARKER_LW, alpha=0.95, zorder=5,
                   label=_grp)

    at.pl.add_lines(ax, intercepts=[-LFC_THRESH, LFC_THRESH], linetype="vline",
                    linestyle="--", color=REF_COLOR, linewidth=THIN_LW)
    at.pl.add_lines(ax, intercepts=[-np.log10(FDR_THRESH)], linetype="hline",
                    linestyle="--", color=REF_COLOR, linewidth=THIN_LW)

    xmin, xmax = result["log2fc"].min(), result["log2fc"].max()
    # Legend below the axes, two columns: inside the axes no corner is empty
    # enough for five entries (the lower left, the emptiest, is about 10 x 8 mm,
    # the key 20 x 13 mm). Markers 3.5 pt, larger than the dots, so that the legend
    # markers can be told apart from the data dots.
    _handles = [Line2D([], [], ls="none", marker="o", markersize=LEG_DOT,
                       markerfacecolor=PHOSPHO_GROUP_COLORS[g], markeredgecolor="black",
                       markeredgewidth=MARKER_LW, label=g)
                for g in GROUPS if (result["group"] == g).any()]
    _leg = fig.legend(handles=_handles, loc="outside lower center", ncol=2, frameon=False,
                      borderaxespad=0, handlelength=0.6, handletextpad=0.3,
                      columnspacing=1.0)

    # --- Labels: anchored columns in the side gutters (alphapepttools) ------
    # label_plot bins each point to the nearest x anchor and stacks the labels
    # in a column there, drawing a straight line to the point, so labels in one
    # column cannot overlap each other. Left and right are separate calls so
    # each column gets its own start height: the left one has to begin below
    # the legend.
    _lab = result[result["gene"].isin(LABEL_NORM)].copy()
    _n_before = len(_lab)
    _lab = _lab.sort_values("fdr").drop_duplicates("gene")
    if len(_lab) < _n_before:
        print(f"  {_n_before - len(_lab)} extra sites on LABEL genes not named "
              f"(kept the most significant site per gene)")
    _lab["label"] = [
        SITE_LABELS.get((r.gene, r.aa, int(r.position)), r.gene)
        for r in _lab.itertuples()
    ]

    _rng = xmax - xmin
    ax.set_xlim(xmin - GUTTER_L * _rng, xmax + GUTTER_R * _rng)
    _anchor_l = xmin - ANCHOR_L * _rng
    _anchor_r = xmax + ANCHOR_R * _rng
    _xmid = float(np.median(result["log2fc"]))

    # label_plot turns its pixel spacing into data units, so the axes must have
    # their final size first: constrained layout sets it at the first draw, once
    # the legend and the axis labels exist. The lines label_plot draws would
    # widen the limits, which are set back afterwards.
    label_axes(ax, xlabel=xlabel, ylabel="−log10 FDR")
    _xlim, _ylim = ax.get_xlim(), ax.get_ylim()
    fig.canvas.draw()

    _before = list(ax.texts)
    for _side, _sub, _anchor, _start in (
        ("right", _lab[_lab["log2fc"] >= _xmid], _anchor_r, LAB_START_R),
        ("left",  _lab[_lab["log2fc"] <  _xmid], _anchor_l, LAB_START_L),
    ):
        if _sub.empty:
            continue
        _label_column(
            ax, _sub, _anchor, _start,
            # gene labels as in the Autophagy volcano: normal weight, white
            # round box, solid leader lines
            label_kwargs=dict(fontsize=ANNOT_FS, zorder=7,
                              bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                                        edgecolor="none", alpha=0.85)),
            line_kwargs=dict(color=REF_COLOR, linewidth=THIN_LW, zorder=6),
        )
        print(f"  {_side} column: {len(_sub)} label(s) at x={_anchor:.2f} "
              f"({', '.join(_sub['label'])})")
    _new = [t for t in ax.texts if t not in _before]
    ax.set_xlim(_xlim)
    ax.set_ylim(_ylim)

    # --- CHECKS: labels must clear the legend and stay inside the axes ------
    fig.canvas.draw()
    _leg_bb = _leg.get_window_extent()
    _ax_bb = ax.get_window_extent()
    _bbs = [(t.get_text(), t.get_window_extent()) for t in _new]
    _hit_leg = [t for t, bb in _bbs if bb.overlaps(_leg_bb)]
    if _hit_leg:
        print(f"  CHECK labels overlapping the legend: {_hit_leg} — lower "
              f"LAB_START_L, or move the legend")
    _hit_edge = [t for t, bb in _bbs
                 if bb.x1 > _ax_bb.x1 or bb.x0 < _ax_bb.x0
                 or bb.y1 > _ax_bb.y1 or bb.y0 < _ax_bb.y0]
    if _hit_edge:
        print(f"  CHECK labels running past the axes: {_hit_edge} — widen "
              f"GUTTER_L / GUTTER_R, or lower LAB_START_R")
    if not (_hit_leg or _hit_edge):
        print(f"  CHECK all {len(_new)} labels inside the axes and clear of "
              f"the legend")

    save_panel(fig, letter, f"volcano_{FILTER_LABEL}_{name}_groups", folder)

# === Both panels: sites that move ===========================================
print("\n" + "=" * 72)
print("  SUMMARY — sites moving in each panel (FDR < "
      f"{FDR_THRESH}, |log2FC| > {LFC_THRESH})")
for _name, _c in volcano_counts.items():
    print(f"    {_c['title']:<46} {_c['up']:5d} enriched  {_c['down']:5d} depleted")
print("=" * 72)

# 10. Sites detected in EA and absent from RC (Figure S9 g, h)

In [ ]:
# ============================================================
# Detection table: phosphosites present in excised aggregates and absent from
# rest cells, per BafA1 state. These sites have no quantifiable RC value and
# cannot enter the comparison of section 9.
#
#   Detection is counted on the OBSERVED matrix (adata_phos_site_obs_raw), before
#   imputation: an imputed value is a value that was not measured, so counting it
#   would make the "absent in RC" side of the selection meaningless.
#
#   One row per VAR KEY, which is site x multiplicity. Collapsing the
#   multiplicities of a site before the selection inflates every count, and it
#   inflates the RC count most for the abundant sites, which then fail the
#   "absent in RC" test. A representative is chosen per site only AFTER the
#   selection, so a site passes if any of its phospho-forms passes.
#
#   Selection: detected in >= 3 EA replicates, in <= 1 RC replicate. Blocks of
#   the rows: "Constitutive" = present (>= 50 % of replicates) in PBS CWO;
#   "PFF-induced" = present in PFF CWO or PFF CW; "Aggregate-exclusive" = neither.
#
#   This is a presence table: no effect size, no test. Absence of detection in
#   DIA is not evidence of absence. The selection is conservative in the
#   direction that matters, because EA is the shallower fraction: a site must
#   clear 3 of 5 in the shallow fraction while the deep one misses it. The list
#   is therefore a lower bound.
# ============================================================
for _n in ("adata_phos_site_obs_raw", "biological", "FRACTION_SHORT",
           "output_dir", "today"):
    if _n not in globals():
        raise NameError(f"{_n!r} is not defined. Run section 2 (AnnData objects "
                        f"in 2.6) and the style cell of section 3 first.")

PS_SRC = adata_phos_site_obs_raw          # Class I sites, observed, not normalised

PS_FRACTIONS = ["PBS Cells Without Aggregates", "PFF Cells Without Aggregates",
                "PFF Cells With Aggregates",    "PFF Pure Aggregates",
                "PFF Rest Cells"]
PS_A = "PFF Pure Aggregates"    # the fraction a site must be present in (EA)
PS_B = "PFF Rest Cells"         # the fraction it must be absent from

PS_MIN_A_N = 3      # detected in at least this many EA replicates
PS_MAX_B_N = 1      # detected in at most this many RC replicates

PS_GROUP_T = 0.5    # fraction of replicates that counts as "present"
PS_GROUPS = ["Constitutive", "PFF-induced", "Aggregate-exclusive"]

# 'PFF Pure Aggregates' -> 'PFF EA', from the STYLING short names
PS_SHORT = {f: short_label(f"-BafA1 {f}").rsplit(" ", 1)[0] for f in PS_FRACTIONS}

# --- one row per var key; the label drops the multiplicity for display ----
_v = PS_SRC.var
PS_LABEL = (_v["Genes_first"].astype(str) + "-" + _v["aa"].astype(str)
            + _v["position"].astype(int).astype(str))
_detected = pd.DataFrame(~np.isnan(PS_SRC.X), index=PS_SRC.obs_names,
                         columns=PS_SRC.var_names)

psite_addendum = {}
for _baf, _title_baf in (("-BafA1", "−BafA1"), ("+BafA1", "+BafA1")):
    _wells, _n_rep = {}, {}
    for _f in PS_FRACTIONS:
        _cond = f"{_baf} {_f}"
        _w = PS_SRC.obs_names[PS_SRC.obs[biological].astype(str) == _cond]
        if len(_w) == 0:
            raise ValueError(f"no samples for condition {_cond!r}; check that "
                             f"it is spelled as in obs[{biological!r}]")
        _wells[_f], _n_rep[_f] = list(_w), len(_w)

    _count = pd.DataFrame({_f: _detected.loc[_wells[_f]].sum(axis=0)
                           for _f in PS_FRACTIONS})
    _n_rep = pd.Series(_n_rep)[PS_FRACTIONS]

    # --- selection, on the keys ---
    _keep = (_count[PS_A] >= PS_MIN_A_N) & (_count[PS_B] <= PS_MAX_B_N)
    _count = _count[_keep]

    # --- one representative per site: the best-detected multiplicity ---
    _lab = PS_LABEL.reindex(_count.index)
    _pick = (_count.assign(_lab=_lab.to_numpy(), _t=_count.sum(axis=1))
                   .sort_values([PS_A, "_t"], ascending=False)
                   .drop_duplicates("_lab"))
    _count = _count.loc[_pick.index]
    _count.index = _pick["_lab"].to_numpy()
    _count = _count.sort_index()
    _frac = _count.div(_n_rep, axis=1)

    # --- row blocks ---
    _pbs = _frac["PBS Cells Without Aggregates"]
    _pff_cells = _frac[["PFF Cells Without Aggregates",
                        "PFF Cells With Aggregates"]].max(axis=1)
    _group = pd.Series(np.where(_pbs >= PS_GROUP_T, PS_GROUPS[0],
                       np.where(_pff_cells >= PS_GROUP_T, PS_GROUPS[1],
                                PS_GROUPS[2])), index=_frac.index)

    # --- row order: block, then EA fraction, then the rest, then the name ---
    _other = _frac[[f for f in PS_FRACTIONS if f != PS_A]].mean(axis=1)
    _ord = pd.DataFrame({"g": pd.Categorical(_group, PS_GROUPS, ordered=True),
                         "a": -_frac[PS_A], "o": -_other, "n": _frac.index})
    _order = _ord.sort_values(["g", "a", "o", "n"]).index
    _count, _frac, _group = _count.loc[_order], _frac.loc[_order], _group.loc[_order]

    psite_addendum[_baf] = dict(count=_count, frac=_frac, group=_group,
                                n_rep=_n_rep, title_baf=_title_baf)

    _sizes = _group.value_counts().reindex(PS_GROUPS).fillna(0).astype(int)
    print(f"{_title_baf}: {len(_frac)} sites pass "
          f"({PS_SHORT[PS_A]} >= {PS_MIN_A_N}/{_n_rep[PS_A]}, "
          f"{PS_SHORT[PS_B]} <= {PS_MAX_B_N}/{_n_rep[PS_B]}) — "
          + ", ".join(f"{g} {_sizes[g]}" for g in PS_GROUPS))

_csv = pd.concat([d["count"].assign(BafA1=b, group=d["group"],
                                    site=d["count"].index)
                  for b, d in psite_addendum.items()], ignore_index=True)
_csv.to_csv(f"{output_dir}/{today}_psites_EA_present_RC_absent.csv", index=False)
print(f"written: {today}_psites_EA_present_RC_absent.csv")

In [ ]:
# ============================================================
# Figure S9g (−BafA1) and S9h (+BafA1), 1 unit wide: heatmap of the detection
# table, one figure per BafA1 state. Cell colour = fraction of replicates in
# which the site was measured, with the count printed in every cell. Colour means
# detection frequency and nothing else: there is no effect size and no test in
# this panel.
# ============================================================
for _n in ("psite_addendum", "PS_FRACTIONS", "PS_SHORT", "PS_GROUPS",
           "PS_A", "PS_B", "PS_MIN_A_N", "PS_MAX_B_N"):
    if _n not in globals():
        raise NameError(f"{_n!r} is not defined. Run the detection-table cell above.")

# The panel grows with the row count, in whole units of the page grid: one unit
# wide, as many units tall as the rows need (at most the page height).
PS_ROW_MM   = 1.9    # printed height of one row
PS_FIXED_MM = 18     # printed height of the column labels and the colourbar
# Site labels and cell counts print at 5 pt: at 6 pt a row needs 2.5 mm, and the
# +BafA1 list would be taller than the 167 mm page.
PS_LAB_FS = native_pt(5.0)
PS_ANN_FS = native_pt(5.0)
PS_GAP_PT = 14      # points between the site labels and the block labels
# White through the CWO blue to a darker blue. Low values stay white, which is
# what "not detected" should look like.
PS_CMAP = LinearSegmentedColormap.from_list(
    "detection", ["#ffffff", FRACTION_COLORS["Cells Without Aggregates"], "#31688e"])
PS_GRID = "#b0b0b0"
PS_SEP_LW = AXES_LW     # block separator width
PS_LETTERS = {"-BafA1": "g", "+BafA1": "h"}

for _baf, _d in psite_addendum.items():
    _frac, _count, _group, _n_rep = _d["frac"], _d["count"], _d["group"], _d["n_rep"]
    _n = len(_frac)
    if _n == 0:
        print(f"{_baf}: no sites pass the selection, no figure drawn")
        continue

    _need_mm = _n * PS_ROW_MM + PS_FIXED_MM
    _units_need = int(np.ceil((_need_mm + GAP_MM) / (UNIT_H_MM + GAP_MM)))
    _units = min(GRID_ROWS, _units_need)
    if _units_need > GRID_ROWS:
        print(f"!! {_n} rows need {_need_mm:.0f} mm, more than the page height; "
              f"the rows are shorter than {PS_ROW_MM} mm")
    fig, axm = create_figure(1, 1, figsize=figsize(1, _units))
    ax = axm.next()
    ax.grid(False)

    _M = _frac[PS_FRACTIONS].to_numpy()
    _mesh = ax.pcolormesh(np.arange(len(PS_FRACTIONS) + 1), np.arange(_n + 1),
                          _M, cmap=PS_CMAP, vmin=0, vmax=1,
                          edgecolors=PS_GRID, linewidth=THIN_LW)
    ax.invert_yaxis()

    # Text colour follows the luminance of the cell, computed from the colormap,
    # so it stays correct whichever ramp PS_CMAP holds.
    for _i in range(_n):
        for _j, _f in enumerate(PS_FRACTIONS):
            _r, _g, _b, _ = PS_CMAP(_M[_i, _j])
            _lum = 0.299 * _r + 0.587 * _g + 0.114 * _b
            ax.text(_j + 0.5, _i + 0.5,
                    f"{int(_count.iloc[_i][_f])}/{int(_n_rep[_f])}",
                    ha="center", va="center", fontsize=PS_ANN_FS,
                    color="white" if _lum < 0.5 else "black")

    ax.set_xticks(np.arange(len(PS_FRACTIONS)) + 0.5)
    ax.set_xticklabels([PS_SHORT[f] for f in PS_FRACTIONS],
                       rotation=30, ha="right", rotation_mode="anchor")
    ax.xaxis.set_ticks_position("bottom")
    ax.set_yticks(np.arange(_n) + 0.5)
    ax.set_yticklabels(_frac.index, fontsize=PS_LAB_FS)
    ax.tick_params(length=0)
    for _s in ax.spines.values():
        _s.set_visible(False)
    # No title: the BafA1 state is the x label.
    ax.set_xlabel(_d["title_baf"])

    # Block separators
    _bounds, _seen = [], 0
    for _g in PS_GROUPS:
        _k = int((_group == _g).sum())
        if _k:
            _bounds.append((_g, _seen, _seen + _k))
            _seen += _k
    for _g, _lo, _hi in _bounds[:-1]:
        ax.axhline(_hi, color="black", linewidth=PS_SEP_LW)

    # Block labels, offset from the axes by the width of the widest site label.
    # Measured on a separate figure and applied in points, so the offset does not
    # depend on when constrained layout settles the axes.
    _tmp = plt.figure(figsize=(4, 4)); _rd = _tmp.canvas.get_renderer()
    _w_pt = max(_tmp.text(0, 0, str(s), fontsize=PS_LAB_FS)
                .get_window_extent(_rd).width for s in _frac.index) / _tmp.dpi * 72
    plt.close(_tmp)
    _off = -(_w_pt + plt.rcParams["ytick.major.pad"] + PS_GAP_PT)
    for _g, _lo, _hi in _bounds:
        ax.annotate(_g.replace("-", "-\n"), xy=(0, (_lo + _hi) / 2),
                    xycoords=("axes fraction", "data"),
                    xytext=(_off, 0), textcoords="offset points",
                    rotation=90, ha="center", va="center",
                    fontweight="bold", fontsize=LABEL_FS)

    # Colourbar below the heatmap: on the right it takes the width the five count
    # columns need ("3/5" is 2.5 mm wide at 5 pt). fraction and aspect keep the
    # bar about 1.5 mm thick at every panel height.
    _cb = fig.colorbar(_mesh, ax=ax, location="bottom", shrink=0.8,
                       fraction=0.015, aspect=12, pad=0.01)
    # Two lines: centred under the heatmap, one line runs past the right edge.
    _cb.set_label("Fraction of replicates\ndetected")

    # No fig.tight_layout(): create_figure uses constrained layout and a
    # colorbar exists, so switching layout engines raises RuntimeError.
    _tag = "MinusBaf" if _baf.startswith("-") else "PlusBaf"
    save_panel(fig, PS_LETTERS[_baf], f"heatmap_EA_present_RC_absent_{_tag}", SI9_DIR)
    _row_mm = ax.get_window_extent().height / fig.dpi * MM_PER_IN * SCALE / _n
    print(f"{_d['title_baf']}: {_n} rows x {len(PS_FRACTIONS)} fractions, "
          f"figsize(1, {_units}), rows {_row_mm:.2f} mm; blocks "
          + ", ".join(f"{g} {hi - lo}" for g, lo, hi in _bounds))
    if _row_mm < 1.8:
        print(f"  !! rows are {_row_mm:.2f} mm, less than the 1.8 mm 5 pt labels need")

# 11. SNCA C-terminal phosphopeptide (Figure 6d, Figure S10a)

In [ ]:
# Figure 6d (−BafA1) and Figure S10a (+BafA1), 1 x 1 unit each: the SNCA
# C-terminal phosphopeptide (residues 103-140, any localisation) per well and
# condition. All its phosphorylated variants in the modified-peptide matrix
# (observed, normalised to the condition mean of SNCA) are summed in linear
# space and returned to log2, one value per well.

def snca_cterm_signal(adata, tag="EMPSEE"):
    """Per sample: log2 of the summed linear signal of every SNCA phosphopeptide
    whose sequence contains `tag` (the C-terminal tryptic peptide), and how many
    of those peptides were measured. Returns (peptide keys, signal, n measured).
    A variant that was not measured counts as zero in the sum."""
    keys = [k for k in adata.var_names
            if str(adata.var.loc[k, "Genes_first"]) == "SNCA"
            and tag in str(adata.var.loc[k, "clean_sequence"])]
    if not keys:
        raise ValueError(f"no SNCA peptide in the layer contains {tag!r}")
    X = np.vstack([np.asarray(adata[:, k].X, dtype=float).ravel() for k in keys])
    linear = np.nansum(2.0 ** X, axis=0)
    signal = np.log2(np.where(linear > 0, linear, np.nan))
    return keys, signal, np.sum(~np.isnan(X), axis=0)


adata = adata_phos_peptide_obs_norm
LAYER_LABEL = "log2 phospho stoichiometry"

ctermsnca, combined_signal, _n_var = snca_cterm_signal(adata)
print(f"Combining {len(ctermsnca)} C-term peptide variants")
print(f"C-term variants measured per sample: "
      + ", ".join(f"{k} variant(s) in {int((_n_var == k).sum())} samples"
                  for k in range(len(ctermsnca) + 1)))


# --- long format: one row per well ---
df_plot = pd.DataFrame({
    "well":      adata.obs.index,
    "condition": adata.obs[biological].astype(str).values,
    "signal":    combined_signal,
})
df_plot = df_plot.dropna(subset=["signal"])

# --- condition order from the style cell, split by BafA1 ---
if "biological_order" not in globals():
    raise NameError("biological_order is not defined. Run the STYLING cell above.")
minus_order = [c for c in biological_order if c.startswith("-BafA1")]
plus_order  = [c for c in biological_order if c.startswith("+BafA1")]

S_DOT = native_pt(2.0) ** 2   # sample dots, 2.0 pt across in print

# (conditions, file tag, x label, panel letter, folder)
panels = [
    (minus_order, "-BafA1", "−BafA1", "d", FIG6_DIR),
    (plus_order,  "+BafA1", "+BafA1", "a", SI10_DIR),
]

for cond_order, file_tag, title_text, letter, folder in panels:
    sub = df_plot[df_plot["condition"].isin(cond_order)].copy()
    sub["condition"] = pd.Categorical(sub["condition"], categories=cond_order, ordered=True)

    fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
    ax = axm.next()

    positions = np.arange(len(cond_order)) * 1.0
    bp_data = [sub[sub["condition"] == c]["signal"].dropna().values for c in cond_order]

    bp = ax.boxplot(bp_data, positions=positions, widths=0.7,
                    patch_artist=True, showfliers=False, zorder=2)
    # The condition colour at full strength with a black edge, as in the
    # Autophagy box plots: an alpha on the patch greys its edge and turns the
    # +BafA1 colours (already half-way to white) white.
    for patch, cond in zip(bp["boxes"], cond_order):
        patch.set_facecolor(biological_color_dict.get(cond, "lightgray"))
        patch.set_edgecolor("black")

    # wells
    for k, cond in enumerate(cond_order):
        vals = bp_data[k]
        if len(vals) == 0:
            continue
        jitter = np.random.default_rng(42).uniform(-0.18, 0.18, size=len(vals))
        ax.scatter(positions[k] + jitter, vals,
                   color=biological_color_dict.get(cond, "gray"),
                   s=S_DOT, edgecolor="black", linewidth=MARKER_LW, zorder=3)

    # tick labels: seeding and fraction on two lines; the BafA1 state is the x label
    ax.set_xticks(positions)
    ax.set_xticklabels([short_label(c).rsplit(" ", 1)[0].replace(" ", "\n") for c in cond_order],
                       fontweight="bold")
    ax.set_xlabel(title_text)
    ax.set_ylabel(LAYER_LABEL)
    ax.set_xlim(positions[0] - 0.6, positions[-1] + 0.6)

    save_panel(fig, letter, f"SNCA_Cterm_phosphopeptide_combined_{file_tag}", folder)

    # mean and median per condition (the fold changes quoted in the text)
    print(f"\n[{title_text}] mean signal per condition:")
    for c in cond_order:
        vals = sub[sub["condition"] == c]["signal"].dropna().values
        if len(vals):
            print(f"  {c}:  n={len(vals)},  mean={np.mean(vals):+.2f},  median={np.median(vals):+.2f}")
        else:
            print(f"  {c}:  (no data)")

# 12. Site assignment from y ions (Figure S10 b–d)

Which acceptor of the SNCA C-terminal peptide carries the phosphate, read from the y ions of the fragment-level export: each y ion adds one acceptor to the stretch it covers, so a phosphate first seen on y13 and not on y8 sits on Ser129.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  SNCA C-terminal y ions: phospho assignment per FRAGMENT. Calculation    ║
# ║  only; the panels are the cells below.                                   ║
# ║                                                                          ║
# ║  A fragment is called phospho only if its own measured m/z is 79.9663    ║
# ║  Th (over its charge) above the theoretical m/z of that fragment in that ║
# ║  oxidation state. The precursor carrying the word "Phospho" says nothing ║
# ║  about which fragment holds it, so the precursor is never used to make   ║
# ║  the call.                                                               ║
# ║                                                                          ║
# ║  Source: the fragment-level Spectronaut export of the same search; the   ║
# ║  precursor report of section 2 has no F.* columns.                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

FRAG_COLS = ["R.FileName", "PEP.StrippedSequence", "EG.ModifiedSequence",
             "F.Charge", "F.FrgLossType", "F.FrgNum", "F.FrgType",
             "F.ExcludedFromQuantification", "F.PeakArea", "F.NormalizedPeakArea",
             "F.CalibratedMz"]

# Fragment intensity carried forward:
#   F.PeakArea           raw integrated area, no cross-run scaling
#   F.NormalizedPeakArea the same area after Spectronaut's run normalisation,
#                        which feeds EG.TotalQuantity and therefore the peptide
#                        matrix of Figure 6d
# The run normalisation equalises the total signal between runs. EA is a
# 2240-shape excision and RC a 560-shape one, so those runs differ in material by
# design, and the raw area is the numerator once the panel is divided by SNCA.
FRAG_AREA_COL = "F.PeakArea"

SNCA_CTERM = "NEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA"   # residues 103-140
SNCA_START = 103
CTERM_TAG  = "EMPSEE"          # unique to the C-terminal tryptic peptide
MZ_TOL     = 0.02              # Th

# y_n starts at residue 141 - n, so each ion adds exactly one new acceptor.
FRAG_IONS  = [5, 8, 13]        # y16 as well, if the search ever reports it
ION_ADDS   = {5: "Y136", 8: "Y133", 13: "S129", 16: "Y125"}
ION_CARRIES = {5:  "Y136",
               8:  "Y133, Y136",
               13: "S129, Y133, Y136",
               16: "Y125, S129, Y133, Y136"}

# monoisotopic residue masses and modification masses, Da
AA_MONO = dict(G=57.02146, A=71.03711, S=87.03203, P=97.05276, V=99.06841,
               T=101.04768, L=113.08406, I=113.08406, N=114.04293, D=115.02694,
               Q=128.05858, K=128.09496, E=129.04259, M=131.04049, H=137.05891,
               F=147.06841, R=156.10111, C=103.00919, Y=163.06333, W=186.07931)
H2O, PROTON, PHOSPHO, OXIDATION = 18.0105646, 1.00727646, 79.9663305, 15.9949146

# --- only the SNCA C-terminal rows of the fragment report ---
_parts = []
for _ch in pd.read_csv(input_frag, sep="\t", usecols=FRAG_COLS,
                       chunksize=2_000_000, low_memory=False):
    _parts.append(_ch[_ch["PEP.StrippedSequence"]
                      .str.contains(CTERM_TAG, regex=False, na=False)])
frag_raw = pd.concat(_parts, ignore_index=True)

def _parse_modified(modseq):
    """Strip Spectronaut bracket modifications; return the plain sequence and
    the 0-based positions carrying Oxidation (M)."""
    s = modseq.strip("_")
    plain, ox, i = [], [], 0
    while i < len(s):
        if s[i] == "[":
            j = s.index("]", i)
            if s[i + 1:j].startswith("Oxidation"):
                ox.append(len(plain) - 1)
            i = j + 1
        else:
            plain.append(s[i]); i += 1
    return "".join(plain), ox

# y ions without neutral loss
_sel = frag_raw[(frag_raw["F.FrgType"] == "y") &
                (frag_raw["F.FrgLossType"] == "noloss")].rename(columns={
    "R.FileName": "file", "EG.ModifiedSequence": "modseq", "F.FrgNum": "ion",
    "F.Charge": "z", FRAG_AREA_COL: "area", "F.CalibratedMz": "mz_obs",
    "F.ExcludedFromQuantification": "excluded"})
n_dropped_type = len(frag_raw) - len(_sel)

# each fragment: unmodified, phospho, ambiguous (both within MZ_TOL) or unassigned
_rows = []
for r in _sel.itertuples(index=False):
    seq, ox = _parse_modified(r.modseq)
    n, z, L = int(r.ion), int(r.z), len(seq)
    frag_seq = seq[L - n:]
    n_ox = sum(1 for p in ox if p >= L - n)
    m_unmod  = sum(AA_MONO[a] for a in frag_seq) + H2O + n_ox * OXIDATION
    mz_unmod = (m_unmod + z * PROTON) / z
    mz_phos  = (m_unmod + PHOSPHO + z * PROTON) / z
    obs = float(r.mz_obs)
    hit_u = abs(obs - mz_unmod) <= MZ_TOL
    hit_p = abs(obs - mz_phos)  <= MZ_TOL
    state = ("ambiguous"  if (hit_u and hit_p) else
             "unmodified" if hit_u else
             "phospho"    if hit_p else "unassigned")
    _rows.append((r.file, n, z, state, float(r.area), bool(r.excluded),
                  obs, mz_unmod, mz_phos))

frag = pd.DataFrame(_rows, columns=["file", "ion", "z", "state", "area",
                                    "excluded", "mz_obs", "mz_unmod", "mz_phospho"])

# --- condition, from the raw file name ---
_COND_RE = re.compile(r"_(MinusInh|PlusInh)_(MinusPFF|PlusPFF)_"
                      r"(CellsWithoutAggs|CellsWithAggs|RestCells|PureAggs)_")
def _fragment_condition(fname):
    m = _COND_RE.search(fname)
    return "_".join(m.groups()) if m else None

frag["condition"] = frag["file"].map(_fragment_condition)

# Sum the phospho-fragment area per (condition, ion, sample). The sum runs over
# every precursor form and charge state that yielded this fragment, which is one
# form for y5 and y8 and nine for y13, so the ions are not drawn from a matched
# precursor set. Each fragment still localises the phospho independently by its
# own m/z, which is what the comparison rests on.
# Fragments that Spectronaut flagged ExcludedFromQuantification are kept: every
# y5 row has that flag, and dropping them would remove the ion from the
# measurement.
area_per_sample = (frag[(frag["state"] == "phospho") & (frag["ion"].isin(FRAG_IONS))]
                   .groupby(["condition", "ion", "file"])["area"].sum()
                   .rename("area").reset_index())

# ══════════════════════════ checks ══════════════════════════
print("=" * 78)
print(f"SNCA C-terminal y ions — per-fragment phospho assignment ({FRAG_AREA_COL})")
print("=" * 78)
print(f"  report rows matching '{CTERM_TAG}'      {len(frag_raw):6d}")
print(f"  not a y ion / neutral loss             {n_dropped_type:6d}  (dropped)")
print(f"  y ions tested against theoretical m/z  {len(frag):6d}")
for _s in ("unmodified", "phospho", "ambiguous", "unassigned"):
    print(f"      {_s:<34s} {int((frag['state'] == _s).sum()):6d}")

print("\n  y-ion coverage of the C-terminal peptide:")
for k in sorted(set(frag["ion"]) | set(ION_ADDS)):
    _s = frag[frag["ion"] == k]
    _adds = ION_ADDS.get(k, "no acceptor")
    if _s.empty:
        print(f"      y{k:<3d} not reported by Spectronaut for any precursor "
              f"(adds {_adds})")
    else:
        print(f"      y{k:<3d} {len(_s):4d} rows  {int((_s['state']=='phospho').sum()):4d} phospho  "
              f"{_s['file'].nunique():2d} samples  "
              f"excluded-from-quant {int(_s['excluded'].sum()):3d}/{len(_s):<4d} (adds {_adds})")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Figure S10b, 1 x 1 unit: which acceptor each y ion newly reports.       ║
# ║  Residues 121-140 of the SNCA 103-140 tryptic peptide. y16 is drawn      ║
# ║  although the search does not report it: it is the ion that would        ║
# ║  separate Y125 from the rest. Shaded: residues 128-132, the window       ║
# ║  between y8 and y13, whose only acceptor is S129.                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

SCHEM_FROM, SCHEM_TO = 121, 140
ACCEPTORS   = {125: "Y", 129: "S", 133: "Y", 136: "Y"}
S129_WINDOW = (128, 132)           # the y8 -> y13 gap; S is its only acceptor
SCHEM_LANES = [16, 13, 8, 5]       # drawn top to bottom
ION_COLORS  = {5: "#9a9a9a", 8: "#555555", 13: HIGHLIGHT_COLOR, 16: "#555555"}
TYR_COLOR, RES_COLOR = "#555555", "#333333"
# Residues and acceptor labels print at 5 pt: in the frame a residue is about
# 1.4 mm wide, an "M" at 6 pt 1.75 mm.
RES_FS    = native_pt(5.0)
ACC_FS    = native_pt(5.0)
ION_BAR_H = 0.07                   # ion lanes, in lane units (about 0.5 mm printed)
SEQ_PAD   = native_pt(1.5)         # frame top to the sequence row, in points

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.grid(False)

# The frame holds the ion lanes: x is the residue position, one row per ion.
# The sequence runs along the top of the frame, the acceptors named above it.
ax.axvspan(S129_WINDOW[0] - 0.5, S129_WINDOW[1] + 0.5,
           color=HIGHLIGHT_COLOR, alpha=0.10, lw=0, zorder=0)
for i, n in enumerate(SCHEM_LANES):
    start, c = 141 - n, ION_COLORS[n]
    # a bar, not a line: the lane is about 0.5 mm thick, above the 1 pt line limit
    ax.barh(i, SCHEM_TO + 0.45 - start, left=start, height=ION_BAR_H,
            color=c, linewidth=0, zorder=2)
    ax.plot([start], [i], marker="|", color=c, ms=native_pt(4.5), mew=AXES_LW, zorder=3)
    ax.annotate(ION_ADDS[n] if n == 5 else f"+ {ION_ADDS[n]}", (1, i),
                xycoords=("axes fraction", "data"), xytext=(native_pt(2.0), 0),
                textcoords="offset points", ha="left", va="center",
                fontsize=LEGEND_FS, color=c)

ax.set_xlim(SCHEM_FROM - 0.6, SCHEM_TO + 0.6)
ax.set_ylim(len(SCHEM_LANES) - 0.4, -0.6)          # y16 at the top
ax.set_yticks(range(len(SCHEM_LANES)))
ax.set_yticklabels([f"y{n}" for n in SCHEM_LANES], fontweight="bold")
for _t, n in zip(ax.get_yticklabels(), SCHEM_LANES):
    _t.set_color(ION_COLORS[n])
ax.set_xticks([])
ax.tick_params(axis="y", length=0)
for _s in ax.spines.values():                      # the frame around the lanes
    _s.set_visible(True)

# The sequence along the top of the frame, acceptors in bold, and the acceptor
# names above it in two rows, every second one higher: "Y133" and "Y136" are
# three residues apart, 4.2 mm, and each label is 4.4 mm wide.
_top = blended_transform_factory(ax.transData, ax.transAxes)
for p in range(SCHEM_FROM, SCHEM_TO + 1):
    a   = SNCA_CTERM[p - SNCA_START]
    acc = p in ACCEPTORS
    col = HIGHLIGHT_COLOR if (acc and a == "S") else TYR_COLOR if acc else RES_COLOR
    ax.annotate(a, (p, 1), xycoords=_top, xytext=(0, SEQ_PAD), textcoords="offset points",
                ha="center", va="bottom", fontsize=RES_FS,
                fontweight="bold" if acc else "normal", color=col)
_acc_y = SEQ_PAD + 1.2 * RES_FS + native_pt(0.5)       # offset of the lower name row
for k, p in enumerate(sorted(ACCEPTORS)):
    a = ACCEPTORS[p]
    ax.annotate(f"{a}{p}", (p, 1), xycoords=_top,
                xytext=(0, _acc_y + (k % 2) * 1.15 * ACC_FS), textcoords="offset points",
                ha="center", va="bottom", fontsize=ACC_FS, fontweight="bold",
                color=HIGHLIGHT_COLOR if a == "S" else TYR_COLOR)

save_panel(fig, "b", "SNCA_Cterm_yion_schematic", SI10_DIR)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  y8 and y13 across the fraction series, starting from the unseeded       ║
# ║  control: PBS CWO -> PFF CWO -> RC -> CW -> EA, per BafA1 state.         ║
# ║  Calculation and summary table; the panels are the cell below.           ║
# ║  Value per fraction: log2 of the mean phospho-fragment area over all     ║
# ║  wells of the condition (an undetected well counts as 0), divided by     ║
# ║  the condition mean of SNCA protein. Well set: prot_filt, so the printed ║
# ║  SNCA row and the drawn lines describe the same wells.                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

FRAG_PANELS = [("MinusInh", "−BafA1", "MinusBaf"),
               ("PlusInh",  "+BafA1", "PlusBaf")]

SNCA_ACCESSION = "P37840"

# y5 is drawn in the schematic and not here: every y5 row has the flag
# F.ExcludedFromQuantification (coverage print of the fragment cell above).
FRAG_PLOT_IONS = [8, 13]

# (condition suffix, short label). PBS is the MinusPFF arm of the same wells.
FRAG_SERIES = [("MinusPFF_CellsWithoutAggs", "PBS\nCWO"),
               ("PlusPFF_CellsWithoutAggs",  "PFF\nCWO"),
               ("PlusPFF_RestCells",         "RC"),
               ("PlusPFF_CellsWithAggs",     "CW"),
               ("PlusPFF_PureAggs",          "EA")]

if SNCA_ACCESSION not in prot_filt.columns:
    raise KeyError(f"{SNCA_ACCESSION} is not a column of prot_filt. "
                   "Run the proteome cells in section 2 first.")

_snca_log2     = prot_filt.set_index("well")[SNCA_ACCESSION].astype(float).dropna()
snca_protein   = 2.0 ** _snca_log2
cond_n_prot    = prot_filt.groupby("condition").size()
_wells_by_cond = prot_filt.groupby("condition")["well"].apply(list)

# Every fragment value divided by the mean SNCA PG.Quantity of its condition, as
# in the normalisation of section 2.5: SNCA comes from the proteome injection,
# so pairing by well does not cancel the loading of the phospho run and adds the
# noise of the proteome run.
_FRAG_WELL_RE = re.compile(r"Phospho_([A-P]\d{2})_")
area_pbs = area_per_sample.copy()
area_pbs["well"] = [_FRAG_WELL_RE.search(f).group(1) for f in area_pbs["file"]]
area_pbs = area_pbs[area_pbs["well"].isin(set(_snca_log2.index))].copy()
_well_cond = prot_filt.set_index("well")["condition"]
_cmean = snca_protein.groupby(_well_cond).mean()
area_pbs["value"] = (area_pbs["area"]
                     / area_pbs["well"].map(_well_cond).map(_cmean))
print(f"fragment samples on the prot_filt well set: {area_pbs['file'].nunique()} "
      f"of {area_per_sample['file'].nunique()}   "
      f"(divided by the condition-mean SNCA; statistic: mean in linear space)")

def _summarise(v, N):
    """log2 of the mean over all N samples of the condition (an undetected
    sample counts as 0), or nan when nothing was detected at all."""
    _val = v.sum() / N if N else np.nan
    return np.log2(_val) if (_val and _val > 0) else np.nan

def _pbs_series(inh):
    """log2 value per fraction for each ion plus SNCA protein, how many samples
    contributed, and the condition n."""
    means, n_det, n_tot = {}, {}, {}
    for k in FRAG_PLOT_IONS:
        m, d, t = [], [], []
        for suf, _lab in FRAG_SERIES:
            key = f"{inh}_{suf}"
            v = area_pbs[(area_pbs.condition == key) & (area_pbs.ion == k)]["value"]
            N = int(cond_n_prot.get(key, 0))
            m.append(_summarise(v, N)); d.append(len(v)); t.append(N)
        means[k], n_det[k], n_tot[k] = np.array(m), np.array(d), np.array(t)
    m, d, t = [], [], []
    for suf, _lab in FRAG_SERIES:
        key = f"{inh}_{suf}"
        _v = snca_protein.reindex(_wells_by_cond.get(key, [])).dropna()
        N = int(cond_n_prot.get(key, 0))
        m.append(np.log2(_v.mean()) if len(_v) else np.nan)
        d.append(len(_v)); t.append(N)
    means["SNCA"], n_det["SNCA"], n_tot["SNCA"] = np.array(m), np.array(d), np.array(t)
    return means, n_det, n_tot

frag_series = {inh: _pbs_series(inh) for inh, _, _ in FRAG_PANELS}

# ------------------------------------------------------------------ summary
print("\n" + "=" * 86)
print("log2 signal across the fraction series, starting from the unseeded control"
      "  (divided by SNCA)")
print("=" * 86)
for inh, title_text, _ in FRAG_PANELS:
    means, n_det, n_tot = frag_series[inh]
    _labs = [lab.replace("\n", " ") for _s, lab in FRAG_SERIES]
    print(f"\n  {title_text}")
    print(f"    {'line':<13s}" + "".join(f"{l:>10s}" for l in _labs))
    # SNCA protein is printed as context. It is not drawn: PG.Quantity of the
    # proteome run is not on the scale of a fragment area.
    for k in FRAG_PLOT_IONS + ["SNCA"]:
        _name = "SNCA protein" if k == "SNCA" else f"y{k}"
        print(f"    {_name:<13s}"
              + "".join("       n.d." if np.isnan(q) else f"{q:10.2f}"
                        for q in means[k]))
        print(f"    {'  detected':<13s}"
              + "".join(f"{d:>7d}/{t:<2d}" for d, t in zip(n_det[k], n_tot[k])))
    _i_ea = len(FRAG_SERIES) - 1
    # The fragments are already divided by SNCA, so their rise is net of the
    # protein; the protein's own rise is printed beside it so that it is not
    # subtracted a second time.
    _rise = ", ".join(
        f"y{k} " + ("n.d. at PBS CWO" if np.isnan(means[k][0])
                    else f"{means[k][_i_ea] - means[k][0]:+.2f}")
        for k in FRAG_PLOT_IONS)
    _snca_rise = means["SNCA"][_i_ea] - means["SNCA"][0]
    print(f"    rise from PBS CWO to EA: {_rise}"
          f"   (already net of SNCA, whose own rise is {_snca_rise:+.2f})")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Figure S10c (−BafA1) and S10d (+BafA1), 1 x 1 unit each: y8 and y13     ║
# ║  across the fraction series, from the cell above. Not anchored, so the   ║
# ║  vertical gap between the two lines is the measured one and it is the    ║
# ║  same in both panels. Open marker: ion not detected in every replicate;  ║
# ║  down triangle on the axis floor: never detected (y8 at PBS CWO, in      ║
# ║  both BafA1 states).                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ION_COLORS comes from the schematic cell (Figure S10b) above.
ION_MARKERS = {5: "o", 8: "s", 13: "D"}
FRAG_LETTERS = {"MinusBaf": "c", "PlusBaf": "d"}
MS_ION = native_pt(3.0)         # ion markers, 3.0 pt across in print

_allp = [frag_series[inh][0][k] for inh, _, _ in FRAG_PANELS for k in FRAG_PLOT_IONS]
_lop  = np.nanmin([np.nanmin(a) for a in _allp])
_hip  = np.nanmax([np.nanmax(a) for a in _allp])
_span = _hip - _lop
# One y range for both panels, from the data. The extra room at the bottom is
# for the "never detected" markers, which are drawn below every real value.
FRAG_YLIM_PBS = (_lop - 0.28 * _span, _hip + 0.08 * _span)
_FLOOR = FRAG_YLIM_PBS[0] + 0.05 * (FRAG_YLIM_PBS[1] - FRAG_YLIM_PBS[0])

for inh, title_text, file_tag in FRAG_PANELS:
    means, n_det, n_tot = frag_series[inh]

    fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
    ax = axm.next()
    x = np.arange(len(FRAG_SERIES))

    for k in FRAG_PLOT_IONS:
        c, mk = ION_COLORS[k], ION_MARKERS[k]
        v = means[k]
        _ok = ~np.isnan(v)
        ax.plot(x[_ok], v[_ok], color=c, lw=LINE_LW, zorder=2)
        for xi, yi, d, t in zip(x, v, n_det[k], n_tot[k]):
            if np.isnan(yi):                       # nothing detected at all
                ax.plot([xi], [_FLOOR], marker="v", ms=MS_ION, linestyle="none",
                        color=c, mfc="white", mec=c, mew=LINE_LW, zorder=3)
                continue
            ax.plot([xi], [yi], marker=mk, ms=MS_ION, linestyle="none", color=c,
                    mfc=c if d == t else "white", mec=c, mew=LINE_LW, zorder=3)

    ax.set_ylim(*FRAG_YLIM_PBS)
    ax.set_xticks(x)
    ax.set_xticklabels([lab for _s, lab in FRAG_SERIES], fontweight="bold")
    ax.set_xlim(-0.45, len(FRAG_SERIES) - 0.55)
    # One line; the y ion, the phospho call and the SNCA divisor go into the
    # figure legend.
    ax.set_ylabel("log2 fragment area / SNCA")
    # The BafA1 state is the x label, as in Figure S10a: the tick labels already
    # name the seeding and the fraction.
    ax.set_xlabel(title_text)

    # Key inside the axes, in the first corner with nothing drawn under it. The
    # entries are the ion names only: the acceptors each ion covers are drawn
    # in the schematic (Figure S10b). Markers 3.5 pt, larger than the data
    # markers.
    handles = [Line2D([], [], color=ION_COLORS[k], marker=ION_MARKERS[k],
                      ms=LEG_DOT, lw=LINE_LW, label=f"y{k}")
               for k in FRAG_PLOT_IONS]
    # Every point of every drawn line, sampled along the segments, so that a line
    # crossing the key counts as well as a marker under it.
    _xy = []
    for k in FRAG_PLOT_IONS:
        _v = means[k]
        _ok = ~np.isnan(_v)
        _xs, _ys = x[_ok], _v[_ok]
        for _i in range(len(_xs)):
            _xy.append((_xs[_i], _ys[_i]))
            if _i + 1 < len(_xs):
                _xy += [(_xs[_i] + f * (_xs[_i + 1] - _xs[_i]),
                         _ys[_i] + f * (_ys[_i + 1] - _ys[_i])) for f in np.linspace(0, 1, 40)]
        _xy += [(xi, _FLOOR) for xi, yi in zip(x, _v) if np.isnan(yi)]
    for _loc in ("upper left", "lower right", "lower left", "upper right"):
        _leg = ax.legend(handles=handles, loc=_loc, frameon=False, handlelength=1.5,
                         handletextpad=0.3, borderaxespad=0.3)
        fig.canvas.draw()
        _lb = _leg.get_window_extent()
        _pts = ax.transData.transform(np.array(_xy))
        _n_under = int(sum(_lb.contains(*q) for q in _pts))
        if _n_under == 0:
            break
    print(f"  [{title_text}] CHECK key at {_loc!r}: points of the lines under it "
          f"{_n_under} (must be 0)")

    save_panel(fig, FRAG_LETTERS[file_tag], f"SNCA_Cterm_yion_fromPBS_protnorm_{file_tag}",
               SI10_DIR)

# 13. SNCA-linked kinases and phosphatases (Figure S10 e, f)

In [ ]:
# Kinases and phosphatases with a reported link to α-synuclein, grouped by the
# residue they act on. Assembled by hand from the literature.
ENZYMES = {
    "Ser129 kinases": [
        "PLK1", "PLK2", "PLK3", "PLK4",
        "CSNK1A1", "CSNK1D", "CSNK1E",
        "CSNK2A1", "CSNK2A2", "CSNK2B",
        "GRK2", "GRK3", "GRK5", "GRK6",
        "LRRK2",
    ],
    "Tyr kinases (Y39/Y125/Y133/Y136)": [
        "FYN", "SRC", "LYN", "LCK", "YES1", "HCK", "FGR",
        "ABL1", "ABL2", "SYK", "PTK2B",
    ],
    "other Ser/Thr kinases reported": [
        "GSK3A", "GSK3B", "CDK5", "MAPK1", "MAPK3", "PRKACA",
        "PRKCA", "PRKCB", "PRKCD", "CAMK2D", "CAMK2G", "DYRK1A", "MARK2",
    ],
    "PP2A catalytic and scaffold": [
        "PPP2CA", "PPP2CB", "PPP2R1A", "PPP2R1B",
    ],
    "PP2A regulatory B subunits": [
        "PPP2R2A", "PPP2R2B", "PPP2R2C", "PPP2R2D",
        "PPP2R5A", "PPP2R5B", "PPP2R5C", "PPP2R5D", "PPP2R5E",
    ],
    "PP2A activity regulators": [
        "PTPA", "LCMT1", "PPME1",
    ],
    "PP1": [
        "PPP1CA", "PPP1CB", "PPP1CC",
    ],
    "calcineurin (PP2B)": [
        "PPP3CA", "PPP3CB", "PPP3CC", "PPP3R1",
    ],
    "Tyr phosphatases": [
        "PTPN1", "PTPN6", "PTPN11", "PTPN12", "PTPRA",
    ],
}

# Gene symbols Spectronaut writes under a different name in this FASTA.
ENZ_ALIASES = {"GRK2": ["ADRBK1"], "GRK3": ["ADRBK2"], "PTPA": ["PPP2R4"]}

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Each enzyme of ENZYMES, EA against RC, both BafA1 states: protein log2FC║
# ║  (eBayes on the imputed proteome) with an interval. Calculation only;    ║
# ║  the forest plot is the cell below, which reads fe_rows, fe_groups and   ║
# ║  FE_STATS from here.                                                     ║
# ║                                                                          ║
# ║  Last block: SNCA protein, then its C-terminal phosphopeptides combined  ║
# ║  into one feature (snca_cterm_signal, section 11). The phospho row comes ║
# ║  from the peptide matrix normalised to the protein, so its log2FC is the ║
# ║  change in phosphorylation beyond the protein change in the row above.   ║
# ║                                                                          ║
# ║  The bar and the colour agree by construction. Colour is Benjamini-      ║
# ║  Hochberg significance; a plain 95 % interval excludes zero on rows      ║
# ║  drawn grey (PPP1CC in +BafA1: p 0.0135, q 0.0541), which contradicts    ║
# ║  the colour. The level is therefore placed inside the BH decision        ║
# ║  gap, between the largest p that BH rejects at FDR 0.05 and the smallest ║
# ║  it does not, so a bar excludes zero on exactly the rows drawn in        ║
# ║  colour. m, p* and the level are printed for the figure legend.          ║
# ║                                                                          ║
# ║  The df behind the interval is exact: limma reports the moderated t but  ║
# ║  not its degrees of freedom, so the df is recovered by solving           ║
# ║  p = 2 * sf(|t|, df). It is one number for the whole fit (residual df    ║
# ║  plus the eBayes prior df).                                              ║
# ║                                                                          ║
# ║  A missing row has one of two meanings, labelled separately:             ║
# ║  "not identified" = no protein group for that gene anywhere in the       ║
# ║  proteome; "too sparse to test" = identified, then dropped by the 70 %   ║
# ║  completeness filter or by the test's own completeness rule. Enzymes     ║
# ║  without a result in either BafA1 state are left out of the figure and   ║
# ║  named in the printed summary.                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

FE_CONTRASTS = {                      # tag -> (treatment, control, file tag)
    "−BafA1": ("-BafA1 PFF Pure Aggregates", "-BafA1 PFF Rest Cells", "MinusBaf"),
    "+BafA1": ("+BafA1 PFF Pure Aggregates", "+BafA1 PFF Rest Cells", "PlusBaf"),
}
FE_FDR_CUT   = 0.05
FE_CI        = 0.95          # used only when nothing survives BH

# "Identified at all" is the proteome report BEFORE the completeness filter
# (prot_long, section 2.4). Without it the two missing states cannot be told
# apart, so this raises.
if "prot_long" not in globals():
    raise NameError("prot_long is not defined. Run the proteome cells in "
                    "section 2 first: it is what distinguishes a gene that was "
                    "never identified from one that was identified and filtered.")
FE_IDENTIFIED = set(str(c) for c in prot_long.columns) - {"well", "condition"}

# --- protein group -> gene symbol ---------------------------------------------
# From the search FASTA (fasta_entries, section 2.1) by group_gene; the proteome
# report has no gene column. The reverse map takes every accession of a group,
# so a gene can be traced to its group when the group lists several.
FE_GROUP2GENE = {g: group_gene(g, fasta_entries) for g in FE_IDENTIFIED}
FE_GENE2GROUPS = {}
for _g in FE_IDENTIFIED:
    for _a in _g.split(";"):
        FE_GENE2GROUPS.setdefault(fasta_entries.get(_a, ("", ""))[1], set()).add(_g)

FE_STATS = {}     # tag -> m, df, p*, interval level, n significant

def _fe_add_ci(res, tag):
    """Attach the standard error and the confidence interval, and record the
    multiplicity bookkeeping the legend needs.

    limma's moderated t uses residual df plus the eBayes prior df. That number
    is not in the result table, so it is recovered from the reported (t, p),
    which is exact and constant across features of one fit.

    The reported fdr is verified to be plain Benjamini-Hochberg over all m
    tested features, and p* is the largest p BH still rejects at FE_FDR_CUT."""
    res = res.copy()
    m = len(res)
    _ok = res[(res["p_value"] > 1e-8) & (res["p_value"] < 0.5) & (res["stat"].abs() > 0)]
    if _ok.empty:
        raise ValueError("no feature usable for recovering the degrees of freedom")
    _row = _ok.iloc[len(_ok) // 2]
    _df = _sopt.brentq(
        lambda d: 2 * _sps.t.sf(abs(_row["stat"]), d) - _row["p_value"], 1.0, 5000.0)

    # CHECK the reported fdr is BH across every tested row: recompute it.
    _p = np.sort(res["p_value"].values)
    _rank = np.arange(1, m + 1)
    _bh = np.clip(np.minimum.accumulate((_p * m / _rank)[::-1])[::-1], 0, 1)
    _bh_dev = float(np.abs(_bh - np.sort(res["fdr"].values)).max())
    assert _bh_dev < 1e-9, (
        f"the reported fdr is not Benjamini-Hochberg over all {m} tested rows "
        f"(max deviation {_bh_dev:.2e}); the caption cannot claim BH at m = {m}")

    # The level is placed inside the BH decision gap, between the largest p that
    # BH rejects and the smallest it does not. Setting it to exactly 1 - p*
    # instead puts the boundary row's endpoint on zero: t.ppf is not the exact
    # float inverse of t.sf, so that row's |t| and t* differ by a couple of ulps
    # and its bar ends at -2e-15 and not at 0. The gap is split on a log scale,
    # the scale of p values.
    # The rule matched is the one the colour uses, q < FE_FDR_CUT, not the
    # step-up p <= FE_FDR_CUT * rank / m: the row attaining the step-up has q
    # exactly equal to the cut, so the two rules differ on it.
    _rej = res["fdr"].values < FE_FDR_CUT
    if _rej.any():
        _pstar = float(res["p_value"].values[_rej].max())
        _pacc = res["p_value"].values[~_rej]
        _alpha = (float(np.sqrt(_pstar * _pacc.min())) if len(_pacc)
                  else float(np.nextafter(_pstar, 1.0)))
        _level = 1.0 - _alpha
    else:
        print(f"    NOTE nothing survives BH at {FE_FDR_CUT}; falling back to "
              f"a plain {FE_CI:.0%} interval")
        _pstar, _level = 1.0 - FE_CI, FE_CI

    _tcrit = _sps.t.ppf(0.5 + _level / 2.0, _df)
    res["se"] = (res["log2fc"] / res["stat"]).abs()
    res["ci_lo"] = res["log2fc"] - _tcrit * res["se"]
    res["ci_hi"] = res["log2fc"] + _tcrit * res["se"]

    _excl = (res["ci_lo"] > 0) | (res["ci_hi"] < 0)
    _dis  = int((_excl != (res["fdr"] < FE_FDR_CUT)).sum())
    FE_STATS[tag] = dict(m=m, df=_df, pstar=_pstar, level=_level,
                         n_sig=int((res["fdr"] < FE_FDR_CUT).sum()),
                         disagreements=_dis)
    print(f"    m {m}, df {_df:.3f}, BH p* {_pstar:.5g}, interval level "
          f"{_level:.4%}, t* {_tcrit:.3f}; BH verified to {_bh_dev:.1e}; "
          f"rows where the bar and the colour disagree: {_dis}")
    assert _dis == 0, "matched interval does not reproduce the BH decisions"
    return res

# --- proteome contrasts ------------------------------------------------------
_fe_bio = biological + "_safe"
adata_prot_imp.obs[_fe_bio] = adata_prot_imp.obs[biological].astype(str).map(_sanitize)

fe_prot = {}
for _tag, (_g1, _g2, _ft) in FE_CONTRASTS.items():
    _k, _r = at.tl.diff_exp_ebayes(
        adata=adata_prot_imp, between_column=_fe_bio,
        comparison=(_sanitize(_g1), _sanitize(_g2)))
    _r["gene"] = _r["protein"].astype(str).map(lambda a: FE_GROUP2GENE.get(a, ""))
    print(f"  proteome {_tag}: {len(_r)} proteins tested")
    fe_prot[_tag] = _fe_add_ci(_r, f"proteome {_tag}").drop_duplicates("gene").set_index("gene")

# --- phosphopeptide contrast, normalised to the protein, not imputed ---------
# The SNCA C-terminal variants are combined into one feature with
# snca_cterm_signal (section 11), and that column is appended to the peptide
# matrix before eBayes, so it is moderated against the whole peptide matrix like
# every other row of the figure.
#
# The sum treats a variant that was not measured as zero, so it also depends on
# the number of variants measured. The print below gives that count per group; if the
# two groups differ, part of the fold change is the counting and not the
# phosphorylation.
_ph = adata_phos_peptide_obs_norm
_ph.obs[_fe_bio] = _ph.obs[biological].astype(str).map(_sanitize)

FE_CTERM_KEY = "SNCA__Cterm_combined"
_ct_keys, _ct_combined, _n_var = snca_cterm_signal(_ph)

print(f"SNCA C-term: {len(_ct_keys)} variants combined into one feature")
for _tag, (_g1, _g2, _ft) in FE_CONTRASTS.items():
    for _grp in (_g1, _g2):
        _m = (_ph.obs[biological].astype(str) == _grp).values
        print(f"    {_grp:<38s} variants measured per sample: "
              f"{_n_var[_m].mean():.2f}  (n={int(_m.sum())})")

_var_ct = pd.concat([_ph.var, pd.DataFrame(index=[FE_CTERM_KEY],
                                           columns=_ph.var.columns)])
_ph_aug = ad.AnnData(
    X=np.hstack([np.asarray(_ph.X, dtype=float), _ct_combined[:, None]]),
    obs=_ph.obs.copy(), var=_var_ct)

fe_phos = {}
for _tag, (_g1, _g2, _ft) in FE_CONTRASTS.items():
    _k, _r = at.tl.diff_exp_ebayes(
        adata=_ph_aug, between_column=_fe_bio,
        comparison=(_sanitize(_g1), _sanitize(_g2)))
    print(f"  peptide  {_tag}: {len(_r)} features tested "
          f"(combined row present: {FE_CTERM_KEY in set(_r['protein'])})")
    fe_phos[_tag] = _fe_add_ci(_r, f"peptide {_tag}").set_index("protein")

# --- assemble the rows -------------------------------------------------------
def _pull(table, key, names=None):
    """(lfc, lo, hi, fdr) for one key, or None where it was not tested."""
    for n in (names or [key]):
        if n in table.index:
            _r = table.loc[n]
            return (float(_r["log2fc"]), float(_r["ci_lo"]),
                    float(_r["ci_hi"]), float(_r["fdr"]))
    return None

def _lookup_prot(gene):
    names = [gene] + ENZ_ALIASES.get(gene, [])
    return {t: _pull(fe_prot[t], gene, names) for t in FE_CONTRASTS}

FE_MISS_ABSENT = "not identified"        # no protein group anywhere
FE_MISS_SPARSE = "too sparse to test"    # identified, then dropped

def _fe_missing_label(gene):
    """Why this gene has no result. The two reasons are different claims: the
    first says the proteome never saw it, the second says it saw it and could
    not test it."""
    _accs = set()
    for n in [gene] + ENZ_ALIASES.get(gene, []):
        _accs |= FE_GENE2GROUPS.get(n, set())
    return FE_MISS_SPARSE if (_accs & FE_IDENTIFIED) else FE_MISS_ABSENT

fe_rows, fe_groups, fe_unquant = [], [], []
for _grp, _genes in ENZYMES.items():
    _start = len(fe_rows)
    for _g in _genes:
        _vals = _lookup_prot(_g)
        _miss = _fe_missing_label(_g)
        if all(v is None for v in _vals.values()):
            fe_unquant.append((_g, _miss))
            continue
        fe_rows.append(("protein", _g, _vals, _miss))
    if len(fe_rows) > _start:
        fe_groups.append((_grp, _start, len(fe_rows) - _start))

_start = len(fe_rows)
fe_rows.append(("protein", "SNCA protein", _lookup_prot("SNCA"), FE_MISS_SPARSE))
fe_rows.append(("phospho", "SNCA C-term phospho",
                {t: _pull(fe_phos[t], FE_CTERM_KEY) for t in FE_CONTRASTS},
                FE_MISS_SPARSE))
fe_groups.append(("SNCA", _start, len(fe_rows) - _start))

_n_listed = sum(len(v) for v in ENZYMES.values())
_absent = [g for g, m in fe_unquant if m == FE_MISS_ABSENT]
_sparse = [g for g, m in fe_unquant if m == FE_MISS_SPARSE]
print(f"\n{len(fe_rows)} rows per panel, of {_n_listed} listed enzymes plus the "
      f"two SNCA rows")
print(f"  {FE_MISS_ABSENT}  ({len(_absent)}): {', '.join(_absent) or 'none'}")
print(f"  {FE_MISS_SPARSE} ({len(_sparse)}): {', '.join(_sparse) or 'none'}")
print("  (both sets are left out of the figure)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Figure S10e (−BafA1) and S10f (+BafA1), forest plots of the cell above, ║
# ║  same rows in both. Point = log2FC EA / RC, bar = the interval whose     ║
# ║  level the calculation cell matched to the BH decision, so a bar         ║
# ║  excludes zero exactly on the rows drawn in colour.                      ║
# ║                                                                          ║
# ║  Geometry in absolute inches, with the left margin measured on the row   ║
# ║  labels. No tight_layout: the q labels and the legend are drawn outside  ║
# ║  the axes. m, p* and the interval level are not drawn; the calculation   ║
# ║  cell and the summary at the end of this cell print them for the legend. ║
# ╚══════════════════════════════════════════════════════════════════════════╝
for _need in ("fe_rows", "fe_groups", "FE_STATS", "FE_CONTRASTS", "FE_FDR_CUT",
              "FE_MISS_ABSENT", "FE_MISS_SPARSE", "fe_unquant", "_absent", "_sparse"):
    if _need not in globals():
        raise NameError(f"{_need!r} is not defined. Run the enzyme calculation "
                        f"cell directly above this one first.")

# --- geometry ----------------------------------------------------------------
# Group headings are drawn inside the axes, so the measured left margin only has
# to hold the gene names.
#
# Sizes are stated in printed mm and converted to canvas inches. The panel is a
# whole number of grid units: as tall as the row slots need (at most the page
# height) and as wide as the minimum axes width needs.
# Row labels, the missing-state notes, the q labels and the group headings print
# at 5 pt: at 6 pt a row slot needs 2.5 mm. The x axis and the legend keep the
# style sizes.
def _in(mm):
    """Printed mm -> canvas inches."""
    return mm * DRAW_SCALE / MM_PER_IN

FE_ROW_FS   = native_pt(5.0)
FE_ROW_MM   = 1.9            # printed height of one row slot
FE_MS       = native_pt(2.2) # point markers, 2.2 pt across in print
FE_RIGHT_IN, FE_TOP_IN, FE_BOT_IN = _in(9.0), _in(1.5), _in(15.0)   # q labels; -; x axis + legend
FE_AXMIN_IN = _in(40.0)      # the widest group heading is 33 mm at 5 pt bold
FE_LETTERS  = {"MinusBaf": "e", "PlusBaf": "f"}
_n_rows = len(fe_rows)
_slots  = _n_rows + len(fe_groups)

_tmp = plt.figure(figsize=(4, 4))
_rend = _tmp.canvas.get_renderer()
_wid_in = max(_tmp.text(0, 0, lab, fontsize=FE_ROW_FS)
              .get_window_extent(_rend).width
              for _k, lab, _v, _m in fe_rows) / _tmp.dpi
plt.close(_tmp)
FE_LEFT_IN = _wid_in + _in(1.5)

_units_w = next((u for u in range(1, GRID_COLS + 1)
                 if figsize(u, 1)[0] - FE_LEFT_IN - FE_RIGHT_IN >= FE_AXMIN_IN), GRID_COLS)
_need_h  = FE_TOP_IN + FE_BOT_IN + _slots * _in(FE_ROW_MM)
_units_h = next((u for u in range(1, GRID_ROWS + 1) if figsize(1, u)[1] >= _need_h), None)
if _units_h is None:
    _units_h = GRID_ROWS
    print(f"!! {_slots} row slots need {_need_h / DRAW_SCALE * MM_PER_IN:.0f} mm, more "
          f"than the page height; the slots are shorter than {FE_ROW_MM} mm")
_W, _H = figsize(_units_w, _units_h)
_row_mm = (_H - FE_TOP_IN - FE_BOT_IN) / _slots / DRAW_SCALE * MM_PER_IN
print(f"geometry: left {FE_LEFT_IN / DRAW_SCALE * MM_PER_IN:.1f} mm (measured), "
      f"figsize({_units_w}, {_units_h}) per panel, row slots {_row_mm:.2f} mm")

# y slots: one blank slot per group heading, then its rows
_ypos, _gy, _cursor = {}, {}, 0.0
for _grp, _s, _n in fe_groups:
    _gy[_grp] = _cursor
    _cursor += 1.0
    for _i in range(_s, _s + _n):
        _ypos[_i] = _cursor
        _cursor += 1.0
_TOTAL = _cursor
_flip = lambda y: _TOTAL - y          # so row 0 is at the top

# One x range for both panels, taken from the interval ends and not the points.
_ends = [x for _k, _l, d, _m in fe_rows for v in d.values() if v is not None
         for x in (v[1], v[2])]
_pad = max(0.4, 0.06 * (max(_ends) - min(_ends)))
FE_XLO, FE_XHI = min(_ends) - _pad, max(_ends) + _pad

for _tag, (_g1, _g2, _ftag) in FE_CONTRASTS.items():
    fig = plt.figure(figsize=(_W, _H))
    ax = fig.add_axes([FE_LEFT_IN / _W, FE_BOT_IN / _H,
                       (_W - FE_LEFT_IN - FE_RIGHT_IN) / _W,
                       (_H - FE_TOP_IN - FE_BOT_IN) / _H])

    for _i, (_kind, _lab, _vals, _miss) in enumerate(fe_rows):
        _yb = _flip(_ypos[_i])
        if _i % 2:
            ax.axhspan(_yb - 0.5, _yb + 0.5, color=shade(BG_COLOR, 0.72), zorder=0)
        _v = _vals[_tag]
        if _v is None:
            # Every drawn row is tested in at least one BafA1 state, so a row
            # without a value here is one that is too sparse in this state.
            ax.text(FE_XLO + 0.02 * (FE_XHI - FE_XLO), _yb, f"{FE_MISS_SPARSE} here",
                    va="center", ha="left", fontsize=FE_ROW_FS,
                    color="0.35", style="italic", zorder=4)
            continue
        _lfc, _lo, _hi, _fdr = _v
        _sig = _fdr < FE_FDR_CUT
        _col = (UP_COLOR if _lfc > 0 else DOWN_COLOR) if _sig else BG_COLOR
        ax.plot([_lo, _hi], [_yb, _yb], color=_col, lw=LINE_LW,
                solid_capstyle="butt", zorder=2)
        for _e in (_lo, _hi):                       # interval caps
            ax.plot([_e, _e], [_yb - 0.22, _yb + 0.22], color=_col, lw=LINE_LW, zorder=2)
        ax.plot(_lfc, _yb, "o", ms=FE_MS, color=_col,
                mec="black" if _sig else _col, mew=MARKER_LW if _sig else 0, zorder=4)
        if _sig:
            _q = f"{_fdr:.3f}" if _fdr >= 5e-4 else f"{_fdr:.0e}"
            ax.text(1.02, _yb, f"q {_q}", va="center", ha="left",
                    fontsize=FE_ROW_FS, color=_col, fontweight="bold",
                    clip_on=False,
                    transform=blended_transform_factory(ax.transAxes, ax.transData))

    for _grp, _s, _n in fe_groups:
        _gyy = _flip(_gy[_grp])
        # White box so a heading reaching past x = 0 does not overlap the zero line
        ax.text(0.005, _gyy, _grp, va="center", ha="left", fontsize=FE_ROW_FS,
                fontweight="bold", color="black", zorder=5,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.5),
                transform=blended_transform_factory(ax.transAxes, ax.transData))
        ax.axhline(_gyy - 0.5, color=REF_COLOR, lw=THIN_LW, zorder=1)

    ax.axvline(0, color="black", lw=AXES_LW, zorder=3)
    ax.grid(axis="x", color=BG_COLOR, lw=THIN_LW, alpha=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlim(FE_XLO, FE_XHI)
    _ally = [_flip(v) for v in list(_ypos.values()) + list(_gy.values())]
    ax.set_ylim(min(_ally) - 0.55, max(_ally) + 0.55)
    ax.set_yticks([_flip(_ypos[_i]) for _i in range(_n_rows)])
    ax.set_yticklabels([lab for _k, lab, _v, _m in fe_rows], fontsize=FE_ROW_FS)
    # No title: the BafA1 state ends the x label, in the words of the volcano.
    ax.set_xlabel(f"log2 FC (EA / RC, {_tag})")

    # Markers 3.5 pt, larger than the points, so that the legend markers can be
    # told apart from the data points
    _handles = [
        Line2D([], [], marker="o", ls="-", lw=LINE_LW, ms=LEG_DOT, color=UP_COLOR, mec="black",
               mew=MARKER_LW, label=f"higher in EA (q < {FE_FDR_CUT})"),
        Line2D([], [], marker="o", ls="-", lw=LINE_LW, ms=LEG_DOT, color=DOWN_COLOR, mec="black",
               mew=MARKER_LW, label=f"lower in EA (q < {FE_FDR_CUT})"),
        Line2D([], [], marker="o", ls="-", lw=LINE_LW, ms=LEG_DOT, color=BG_COLOR,
               label="not significant"),
    ]
    _st = FE_STATS[f"proteome {_tag}"]
    # Centred on the canvas, on its bottom edge (borderaxespad=0): centred on the
    # axes, the three columns run past the right edge.
    fig.legend(handles=_handles, loc="lower center", bbox_to_anchor=(0.5, 0.0),
               borderaxespad=0, ncol=3, frameon=False, handlelength=1.5, handletextpad=0.4,
               columnspacing=1.0,
               title=f"point = log2 FC, bar = {_st['level']:.1%} interval")

    # ------------------------------------------------------------- CHECKS
    fig.canvas.draw()
    _over = [t for t in ax.texts if t.get_text().startswith("q ")
             and t.get_window_extent().x1 > fig.bbox.x1 - 2]
    _lb = fig.legends[0].get_window_extent()
    _ylab = max(t.get_window_extent().width for t in ax.get_yticklabels())
    print(f"CHECK [{_tag}] q labels past the canvas edge {len(_over)} (must be 0); "
          f"legend y0 {_lb.y0:.0f} (must be >= 0); widest row label {_ylab:.0f}px "
          f"of {FE_LEFT_IN / _W * fig.bbox.x1:.0f}px margin")
    if not (_lb.x0 >= -1 and _lb.x1 <= fig.bbox.x1 + 1 and _lb.y0 >= -1):
        print(f"  !! [{_tag}] legend is clipped; raise FE_BOT_IN or lower ncol")
    _xl = ax.xaxis.label.get_window_extent()
    print(f"CHECK [{_tag}] gap between legend top and x label bottom: "
          f"{_xl.y0 - _lb.y1:.0f}px (must be > 0)")
    if _xl.y0 <= _lb.y1:
        print(f"  !! [{_tag}] the legend runs into the x axis label; raise FE_BOT_IN")
    _clip = [(_l, _vals[_tag]) for _k, _l, _vals, _m in fe_rows
             if _vals[_tag] is not None
             and (_vals[_tag][1] < FE_XLO or _vals[_tag][2] > FE_XHI)]
    print(f"CHECK [{_tag}] intervals running past the x limits (must be 0): {len(_clip)}")

    save_panel(fig, FE_LETTERS[_ftag], f"SNCA_enzymes_EA_vs_RC_{_ftag}", SI10_DIR)

# --- printed summary ---------------------------------------------------------
print("\n" + "=" * 86)
print("Excised aggregates vs Rest Cells — log2 fold change [interval] and FDR")
print("  interval levels differ slightly between contrasts; see the bookkeeping below")
print("=" * 86)
for _grp, _s, _n in fe_groups:
    print(f"\n  {_grp}")
    for _i in range(_s, _s + _n):
        _kind, _lab, _vals, _miss = fe_rows[_i]
        _txt = []
        for _t in FE_CONTRASTS:
            _v = _vals[_t]
            _txt.append(f"{_t} {_miss}" if _v is None else
                        f"{_t} {_v[0]:+.2f} [{_v[1]:+.2f}, {_v[2]:+.2f}] q {_v[3]:.3f}")
        print(f"    {_lab:<22s} " + "   ".join(_txt))
for _lab, _set in ((FE_MISS_ABSENT, _absent), (FE_MISS_SPARSE, _sparse)):
    if _set:
        print(f"\n  {_lab} ({len(_set)}): {', '.join(_set)}")
print("\n  multiplicity bookkeeping, for the caption:")
for _k, _v in FE_STATS.items():
    print(f"    {_k:<20s} m {_v['m']:5d}  df {_v['df']:.3f}  BH p* {_v['pstar']:.5g}"
          f"  interval level {_v['level']:.4%}  significant {_v['n_sig']}")

# 14. Figure assembly

## 14.1 Figure 6

In [ ]:
# Figure 6 on the 4-column page grid: one letter per unit column, "." empty.
# All four panels are 1 x 1.
figure_name = f"{today}_Figure6_Nanophos"
layout = "a b . .\n c d . ."

compose_figure(
    folder=FIG6_DIR,
    layout=layout,
    output=f"{FIG6_DIR}/{figure_name}.svg",
    show_box=True,
    panel_offset_mm={"b": (2, 0),
                     "d": (2, 0)}
)

## 14.2 Figure S9

In [ ]:
# Figure S9 (panels in SI9_DIR) on the 4-column page grid.
figure_name = f"{today}_SI9_Nanophos"
layout = "a a c h\n b b g h\n d e g h\n f . g h"

compose_figure(
    folder=SI9_DIR,
    layout=layout,
    output=f"{SI9_DIR}/{figure_name}.svg",
    show_box=True,
)

## 14.3 Figure S10

In [ ]:
# Figure S10 (panels in SI10_DIR) on the 4-column page grid.
figure_name = f"{today}_SI10_Nanophos_SNCA"
layout = "a b c d\n e e f f\n e e f f\n e e f f"

compose_figure(
    folder=SI10_DIR,
    layout=layout,
    output=f"{SI10_DIR}/{figure_name}.svg",
    show_box=True,
    panel_offset_mm={"a": (0, 2),
                     "c": (0, 2),
                     "d": (0, 2)}
)